# Structural aliasing in Chronos-Bolt - Bayesian analysis

**PATCHALIAS / Deliverable 3.** This notebook is the executable analysis companion to
`coursework/deliverable3/`. Its inferential population is the frozen set of 15 uniformly trained
patch/stride configurations in `tab:hfModels`; every context has 480 samples and closes on a stride
boundary.

| section | Deliverable 3 analysis | claim | model |
|---|---|---|---|
| 1.2, 3.2, 4.1, 5 | paired local recovery contrast, Eq. (8)-(9) | H1 behavioural, M1 | A / A' |
| 1.3, 3.2, 4.2, 5 | frequency-local MDL codelength, Eq. (10) | H1 representational | B |
| 1.4, 3.2, 4.3, 5 | phase-bin spread, Eq. (11) | H2 | C |
| 1.5, 3.2, 4.4, 5 | collapse location on the two grids, Eq. (12) | H3 location | D1 |
| 1.5, 3.2, 4.5, 5 | branch fundamental and gap movement, Eq. (13) | H3a/H3b | D2 |

The five parts are priors, Chronos observation, likelihood/recovery, posterior inference, and
scientific validity checks. Collection shards, generated signals, model checkpoints and posterior
artifacts are atomically written and fingerprinted. A stale or mixed artifact stops the notebook.

Generated-background results (TSMixup and KernelSynth) are the primary evidence. Pure sinusoids
are an exact-degeneracy reference/positive control. Smoke and synthetic parameter-recovery output
validate the pipeline only and are never empirical evidence for a Deliverable 3 claim.

## 0.1 - Repository

Locate the checkout containing this notebook. A fresh hosted runtime clones the repository first,
so the next cell can install the exact environment recorded in the committed `uv.lock`.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/FedericoSabbadini/patchAliasing.git"
MARKER = Path("chronos") / "bayesian" / "probe_lib.py"

def find_repo() -> Path:
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / MARKER).is_file():
            return candidate
    target = here / "patchAliasing"
    if not (target / MARKER).is_file():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(target)])
    if not (target / MARKER).is_file():
        raise FileNotFoundError(f"{MARKER} is missing from {target}")
    return target

REPO = find_repo()
BAYES_DIR = REPO / "chronos" / "bayesian"
if str(BAYES_DIR) not in sys.path:
    sys.path.insert(0, str(BAYES_DIR))
print("repository:", REPO)
print("modules   :", BAYES_DIR)

## 0.2 - Locked environment

Install the environment exported from the committed `uv.lock`, with the Python-version markers in
that lock. This prevents an unattended long run from silently resolving a different PyMC, ArviZ,
Chronos or parquet stack. A runtime restart may be needed after the first install.

In [ ]:
import importlib.util, shutil, subprocess, sys, sysconfig, tempfile

def _uv_executable() -> str:
    found = shutil.which("uv")
    if found:
        return found
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv"])
    candidates = [
        Path(sysconfig.get_path("scripts")) / ("uv.exe" if os.name == "nt" else "uv"),
        Path(sys.executable).parent / ("uv.exe" if os.name == "nt" else "uv"),
    ]
    for candidate in candidates:
        if candidate.is_file():
            return str(candidate)
    raise FileNotFoundError("uv was installed but its executable was not found")

UV = _uv_executable()
with tempfile.TemporaryDirectory() as temporary:
    requirements = Path(temporary) / "requirements.locked.txt"
    subprocess.check_call([
        UV, "export", "--frozen", "--no-dev", "--no-emit-project", "--no-hashes",
        "--output-file", str(requirements),
    ], cwd=REPO)
    subprocess.check_call([
        UV, "pip", "install", "--python", sys.executable, "--requirement", str(requirements)
    ])

def _real_module(name: str) -> bool:
    try:
        spec = importlib.util.find_spec(name)
    except (ImportError, ValueError):
        return False
    return spec is not None and spec.origin is not None

HAVE_TORCH = _real_module("torch") and _real_module("chronos")
print({"locked environment installed": True, "Part 2 available": HAVE_TORCH})
print("If imports fail after this first install, restart the runtime and resume at this cell.")

## 0.3 - Imports, seed and immutable run namespace

The run namespace includes the Deliverable 3 design version. Every artifact is recorded in an
analysis manifest with its SHA-256; a file without a matching entry is not a checkpoint.

In [ ]:
from __future__ import annotations

import json, random
from importlib import metadata as importlib_metadata

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import arviz as az
import pymc as pm
from scipy import stats

import checkpointing as cp
import bayesian_checks as bc
import model_loader as ml

az.style.use("arviz-whitegrid")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
RNG = np.random.default_rng(SEED)

RESUME = True
DRAWS, TUNE, CHAINS = 2000, 2000, 4
TARGET_ACCEPT = 0.9
NUTS_BACKEND = "pymc"       # "nutpie" is allowed; it targets the same posterior

if NUTS_BACKEND != "pymc":
    if importlib.util.find_spec(NUTS_BACKEND) is None:
        raise ImportError(f"{NUTS_BACKEND} is absent from the locked environment")
print("NUTS backend:", NUTS_BACKEND)

USE_DRIVE = True
DRIVE_ROOT = "/content/drive/MyDrive/patchAliasing"
RUN_ID = "d3_15model_v1"

def _run_root() -> Path:
    on_colab = importlib.util.find_spec("google.colab") is not None
    if not on_colab or not USE_DRIVE:
        return BAYES_DIR
    if not Path("/content/drive/MyDrive").exists():
        from google.colab import drive
        drive.mount("/content/drive")
    root = Path(DRIVE_ROOT)
    root.mkdir(parents=True, exist_ok=True)
    return root

_root = _run_root()
CKPT_DIR = _root / "full" / RUN_ID
DATA_DIR = CKPT_DIR / "data"
FIG_DIR = CKPT_DIR / "figures"
for directory in (CKPT_DIR, DATA_DIR, FIG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

def _version(package: str):
    try:
        return importlib_metadata.version(package)
    except importlib_metadata.PackageNotFoundError:
        return None

notebook_document = json.loads((BAYES_DIR / "bayesian_analysis.ipynb").read_text(encoding="utf-8"))
notebook_logic = [
    (item["cell_type"], "".join(item.get("source", ""))) for item in notebook_document["cells"]
]
source_sha256 = {
    name: cp.sha256_file(BAYES_DIR / name)
    for name in ("collect.py", "probe_lib.py", "model_loader.py", "checkpointing.py",
                 "bayesian_checks.py")
}
source_sha256["bayesian_analysis.ipynb:logic"] = cp.fingerprint(notebook_logic)

ANALYSIS_SPEC = {
    "run_id": RUN_ID,
    "deliverable": "coursework/deliverable3",
    "seed": SEED,
    "sampling": {"draws": DRAWS, "tune": TUNE, "chains": CHAINS,
                 "target_accept": TARGET_ACCEPT, "backend": NUTS_BACKEND},
    "checkpoint_repo": ml.SWEEP_REPO,
    "checkpoint_revision": ml.SWEEP_REVISION,
    "source_sha256": source_sha256,
    "uv_lock_sha256": cp.sha256_file(REPO / "uv.lock"),
    "packages": {name: _version(name) for name in
                 ("pymc", "arviz", "pyarrow", "torch", "chronos-forecasting")},
}
ANALYSIS_FINGERPRINT = cp.fingerprint(ANALYSIS_SPEC)
ANALYSIS_MANIFEST_PATH = CKPT_DIR / "analysis_manifest.json"

if ANALYSIS_MANIFEST_PATH.is_file():
    ANALYSIS_MANIFEST = json.loads(ANALYSIS_MANIFEST_PATH.read_text(encoding="utf-8"))
    if ANALYSIS_MANIFEST.get("analysis_fingerprint") != ANALYSIS_FINGERPRINT:
        raise ValueError("analysis manifest mismatch; increment RUN_ID instead of mixing artifacts")
else:
    ANALYSIS_MANIFEST = {
        "schema_version": 1,
        "analysis_fingerprint": ANALYSIS_FINGERPRINT,
        "analysis_spec": ANALYSIS_SPEC,
        "artifacts": {},
    }
    cp.atomic_json(ANALYSIS_MANIFEST_PATH, ANALYSIS_MANIFEST)

def banner(part: str) -> None:
    print("=" * 78)
    print(part)
    print("=" * 78)

def ckpt(name: str) -> Path:
    return CKPT_DIR / name

def _record(name: str) -> None:
    path = ckpt(name)
    ANALYSIS_MANIFEST["artifacts"][name] = {
        "sha256": cp.sha256_file(path), "bytes": path.stat().st_size
    }
    cp.atomic_json(ANALYSIS_MANIFEST_PATH, ANALYSIS_MANIFEST)

def have(name: str) -> bool:
    if not RESUME:
        return False
    path = ckpt(name)
    entry = ANALYSIS_MANIFEST["artifacts"].get(name)
    if path.exists() != (entry is not None):
        raise ValueError(f"untracked or missing analysis artifact: {name}")
    if not path.exists():
        return False
    if cp.sha256_file(path) != entry["sha256"]:
        raise ValueError(f"analysis artifact hash mismatch: {name}")
    return True

def save_idata(idata, name: str):
    cp.atomic_netcdf(ckpt(name), idata)
    _record(name)
    print("  checkpoint ->", name)
    return idata

def load_idata(name: str):
    if not have(name):
        raise FileNotFoundError(name)
    print("  checkpoint <-", name)
    return az.from_netcdf(ckpt(name))

def save_df(frame: pd.DataFrame, name: str):
    cp.atomic_parquet(ckpt(name), frame)
    _record(name)
    print("  checkpoint ->", name)
    return frame

def load_df(name: str) -> pd.DataFrame:
    if not have(name):
        raise FileNotFoundError(name)
    print("  checkpoint <-", name)
    return pd.read_parquet(ckpt(name))

def save_json(value, name: str):
    cp.atomic_json(ckpt(name), value)
    _record(name)
    print("  checkpoint ->", name)
    return value

def cached_fit(name: str, build_and_sample):
    return load_idata(name) if have(name) else save_idata(build_and_sample(), name)

print("checkpoints:", CKPT_DIR)
print("tables     :", DATA_DIR)
print("figures    :", FIG_DIR)
print(f"draws={DRAWS} tune={TUNE} chains={CHAINS} target_accept={TARGET_ACCEPT}")

## 0.4 - Frozen Deliverable 3 design

`probe_lib.DELIVERABLE3_MODELS` is the immutable 15-configuration registry. The environment
variable `PATCHALIASING_MODELS` may choose a collection session subset, but cannot alter the
planned design, union grid or final coverage gate.

In [ ]:
import probe_lib as pl

FULL_MODELS = list(pl.DELIVERABLE3_MODELS)
SESSION_MODELS = list(pl.SESSION_MODELS)
assert len(FULL_MODELS) == 15
assert all(pl.CTX % S == 0 for _, S in FULL_MODELS)

print(f"fs={pl.FS} Hz  context={pl.CTX}  horizon={pl.PRED}  band={pl.BAND} Hz")
print(f"generators={pl.GENERATORS}  full models={len(FULL_MODELS)}  "
      f"session models={len(SESSION_MODELS)}")

rows = []
for P, S in FULL_MODELS:
    offsets = {f: pl.control_offset(P, S, f) for f in pl.f_lock(P, S)}
    usable = [f for f, delta in offsets.items() if np.isfinite(delta)]
    used_delta = [offsets[f] for f in usable]
    rows.append({
        "model": pl.model_tag(P, S), "P": P, "S": S, "overlap": (P - S) / P,
        "context_closes": pl.CTX % S == 0,
        "n_lock": len(offsets), "n_usable": len(usable),
        "stride_only": sum(pl.lock_family(f, P, S) == "stride" for f in offsets),
        "delta_min_hz": min(used_delta), "delta_max_hz": max(used_delta),
    })
design = pd.DataFrame(rows)
display(design)

gaps = pl.design_gaps(SESSION_MODELS)
if gaps:
    print("\nSession limitations (not empirical null results):")
    for gap in gaps:
        print(" -", gap)
print("\nS in {4, 5, 15, 28} is excluded exactly as documented in Deliverable 3; "
      "no excluded checkpoint can enter the final manifest.")

---
# Part 1, Priors

The deliverable's Bayesian section names a prior for every parameter. This part does two things
with them, **before any observation exists**:

1. writes them down in one table, so the reported analysis and the document cannot drift apart;
2. checks what they *imply*. A prior is not weakly informative because it is labelled weakly
   informative, it is weakly informative if the data it predicts are plausible and if it does not
   quietly assert the conclusion. We therefore push each prior through its own likelihood and look
   at the resulting distribution of the quantities we care about: the recovery ratio $e^{\bar\beta}$,
   the codelength ratio $e^{\theta_{lock}}$, the shape of a collapse comb.

Nothing here needs Chronos: the *design* (which geometries, which lock sites, which phases) is
fixed in advance by `probe_lib`, and only the responses are unknown. That is precisely what makes a
genuine prior predictive check possible.

In [ ]:
banner("PART 1, PRIORS")

# Prior scales are preregistered here, including the sensitivity ladder used in Part 5.
PRIOR_SCALE = 0.5                       # the deliverable's primary choice
PRIOR_LADDER = [0.25, 0.5, 1.0]         # sceptical / primary / wide, for the sensitivity refits
NU = 4                                  # Student-t degrees of freedom, as written in the .tex

# Decision thresholds, preregistered so they cannot be chosen after seeing the posterior.
ATTENUATION_20 = np.log(0.8)            # "at least 20% attenuation"  ->  beta < log 0.8
ROPE_LOG = np.log(1.1)                  # practical equivalence: a +/-10% effect is no effect
ROPE_SLOPE = 0.1                        # H3a / H3b movement: |kappa_F - 1| < 0.1

PRIOR_SPEC = pd.DataFrame([
    # model, parameter, prior, role, where it is written in the deliverable
    ("A/C", "beta_bar",      f"StudentT(nu=4, 0, {PRIOR_SCALE})",  "population phase-lock effect on log-recovery", "Eq. (9)"),
    ("A",   "delta_O",       f"StudentT(nu=4, 0, {PRIOR_SCALE})",  "slope in the centred overlap Otilde; M1", "Eq. (9)"),
    ("A",   "delta_P",       f"StudentT(nu=4, 0, {PRIOR_SCALE})",  "slope in the centred log patch size", "Eq. (9)"),
    ("A/C", "tau",           f"HalfStudentT(nu=4, {PRIOR_SCALE})", "spread of configuration effects", "Eq. (9)"),
    ("A/C", "sigma_harm",    f"HalfStudentT(nu=4, {PRIOR_SCALE})", "spread across lock harmonics", "Eq. (9)"),
    ("A/C", "sigma_bg",      f"HalfStudentT(nu=4, {PRIOR_SCALE})", "spread across background realisations", "Eq. (9)"),
    ("A/C", "sigma",         f"HalfStudentT(nu=4, {PRIOR_SCALE})", "residual scale of the contrast", "Eq. (9)"),
    ("C",   "sigma_phase",   "HalfNormal(0.25)",                   "spread of the per-phase offsets (8 phase bins)", "Eq. (11)"),
    ("B",   "alpha_0",       "Normal(log mean(y), 1)",             "baseline log codelength", "Eq. (10)"),
    ("B",   "theta_lock",    "Normal(0, 0.5)",                     "log codelength expansion at a lock", "Eq. (10)"),
    ("B",   "k (shape)",     "Gamma(2, 0.1)",                      "Gamma dispersion", "Eq. (10)"),
    ("D1",  "alpha_g",       "Normal(0, 1)",                       "off-grid level of log z_g", "Eq. (12)"),
    ("D1",  "theta_S",       "Normal(0, 1)",                       "dip on the stride grid; H3 predicts < 0", "Eq. (12)"),
    ("D1",  "theta_P",       "Normal(0, 1)",                       "dip on the patch grid; H3 predicts < 0", "Eq. (12)"),
    ("D1",  "sigma",         "HalfNormal(1)",                      "residual scale", "Eq. (12)"),
    ("D2",  "kappa_S",       "Normal(0, 1)",                       "measured / stride-predicted spacing; H3a predicts 1", "Eq. (13)"),
    ("D2",  "kappa_P",       "Normal(0, 1)",                       "measured / patch-predicted spacing; H3b predicts 1", "Eq. (13)"),
    ("D2",  "sigma_extra",   "HalfNormal(5) Hz",                   "scatter beyond the sweep resolution", "Eq. (13)"),
], columns=["model", "parameter", "prior", "role", "deliverable"])
display(PRIOR_SPEC)

## 1.1, The design skeleton

Which geometries, lock sites, phases, backgrounds and generators the experiment will visit is
fixed by the design, not by the model. Building that skeleton now lets Part 1 simulate responses
from the prior on exactly the rows Part 2 will later fill in, a real prior predictive check on the
real design, not on a stylised one.

In [ ]:
def design_skeleton(n_phase=10, n_bg=6, generators=pl.GENERATORS) -> pd.DataFrame:
    """Compact support skeleton for prior prediction; Part 2 uses 100 backgrounds per generator."""
    rows = []
    for P, S in FULL_MODELS:
        offsets = {f: pl.control_offset(P, S, f) for f in pl.f_lock(P, S)}
        for f_lock in [f for f, delta in offsets.items() if np.isfinite(delta)]:
            for generator in generators:
                for bg_id in range(n_bg):
                    for phase_idx, phase in enumerate(pl.phases_Sf(f_lock, n_phase)):
                        rows.append({
                            "model": pl.model_tag(P, S), "P": P, "S": S,
                            "overlap": (P - S) / P, "f_lock": f_lock,
                            "family": pl.lock_family(f_lock, P, S), "generator": generator,
                            "bg_id": bg_id, "phase_idx": phase_idx, "phase": float(phase),
                        })
    return pd.DataFrame(rows)

SKELETON = design_skeleton(n_phase=10, n_bg=6)
FULL_CONTRAST_ROWS = sum(
    len(pl.phases_Sf(f, 10)) * 100 * len(pl.GENERATORS)
    for P, S in FULL_MODELS
    for f in pl.f_lock(P, S)
    if np.isfinite(pl.control_offset(P, S, f))
)
print(f"prior-predictive support skeleton: {len(SKELETON):,} rows")
print(f"exact full contrast design: {FULL_CONTRAST_ROWS:,} rows (100 backgrounds/generator)")
display(SKELETON.groupby("model").agg(rows=("f_lock", "size"), locks=("f_lock", "nunique")))

## 1.2, Prior predictive for Models A and C (H1 behavioural, H2)

The contrast $d$ lives on a log-ratio scale, so $\bar\beta$ is read as $e^{\bar\beta}$: the
multiplicative change in forecast amplitude recovery at a lock relative to its controls. The
question a prior predictive answers is whether $\text{Student-}t_4(0, 0.5)$ describes *ignorance*
about that ratio or a belief about it.

Three things are checked:

* the prior is symmetric, it gives attenuation and amplification the same mass, so finding
  attenuation cannot be an artefact of the prior;
* it reaches far enough, a scale of $0.5$ puts $e^{\pm 0.5}\approx 1.65$ at one scale unit and the
  heavy $t_4$ tails admit far larger effects, so a real strong effect will not be shrunk away;
* the simulated contrasts $d$ are on the same order as contrasts that could physically occur
  (recovery ratios are bounded below by 0 and rarely exceed a few).

In [ ]:
def simulate_contrast_prior(skeleton: pd.DataFrame, scale: float, n_draw: int = 2000,
                            rng=None) -> dict:
    """Draw parameters from the Model A prior and push them through the likelihood.

    Returns both the population-level effect draws and simulated contrasts on the real design, so
    the prior can be judged on the data it predicts rather than on its own parameters.
    """
    rng = rng or np.random.default_rng(SEED)
    cfg_codes, cfg_index = pd.factorize(skeleton["model"])
    harm_codes, _ = pd.factorize(skeleton["f_lock"].round(3))
    bg_codes, _ = pd.factorize(skeleton["generator"] + "#" + skeleton["bg_id"].astype(str))
    O = skeleton.groupby("model")["overlap"].first().reindex(cfg_index).to_numpy()
    O_t = (O - O.mean()) / 0.5
    P = skeleton.groupby("model")["P"].first().reindex(cfg_index).to_numpy()
    logP_t = np.log(P) - np.mean(np.log(P))

    # priors, exactly as tabulated above
    beta_bar = stats.t.rvs(NU, scale=scale, size=n_draw, random_state=rng.integers(1 << 31))
    delta_O = stats.t.rvs(NU, scale=scale, size=n_draw, random_state=rng.integers(1 << 31))
    delta_P = stats.t.rvs(NU, scale=scale, size=n_draw, random_state=rng.integers(1 << 31))
    half = lambda: np.abs(stats.t.rvs(NU, scale=scale, size=n_draw,
                                      random_state=rng.integers(1 << 31)))
    tau, sig_h, sig_b, sigma = half(), half(), half(), half()

    # one simulated dataset per draw would be huge; simulate a random design row per draw instead,
    # which gives the marginal prior predictive of a single observation
    i = rng.integers(0, len(skeleton), n_draw)
    beta_c = (beta_bar + delta_O * O_t[cfg_codes[i]] + delta_P * logP_t[cfg_codes[i]]
              + tau * rng.normal(size=n_draw))
    mu = beta_c + sig_h * rng.normal(size=n_draw) + sig_b * rng.normal(size=n_draw)
    d = mu + sigma * stats.t.rvs(NU, size=n_draw, random_state=rng.integers(1 << 31))
    return dict(beta_bar=beta_bar, delta_O=delta_O, delta_P=delta_P, d=d, ratio=np.exp(beta_bar))


fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for scale in PRIOR_LADDER:
    sim = simulate_contrast_prior(SKELETON, scale)
    lbl = f"scale {scale}" + ("  (primary)" if scale == PRIOR_SCALE else "")
    axes[0].hist(sim["beta_bar"], bins=80, range=(-3, 3), histtype="step", lw=1.6, label=lbl, density=True)
    axes[1].hist(np.clip(sim["ratio"], 0, 4), bins=80, histtype="step", lw=1.6, label=lbl, density=True)
    axes[2].hist(np.clip(sim["d"], -6, 6), bins=80, histtype="step", lw=1.6, label=lbl, density=True)

axes[0].axvline(ATTENUATION_20, color="crimson", ls="--", lw=1, label="log 0.8 (20% attenuation)")
axes[0].set_title(r"prior on $\bar\beta$ (log recovery ratio)"); axes[0].set_xlabel(r"$\bar\beta$")
axes[1].axvline(1.0, color="k", lw=0.8)
axes[1].set_title(r"implied recovery ratio $e^{\bar\beta}$"); axes[1].set_xlabel("ratio")
axes[2].set_title("prior predictive contrast $d$ on the real design"); axes[2].set_xlabel("$d$")
for a in axes: a.legend(fontsize=7)
fig.tight_layout(); fig.savefig(FIG_DIR / "P1_prior_predictive_contrast.png", dpi=140,
                                bbox_inches="tight"); plt.show()

sim = simulate_contrast_prior(SKELETON, PRIOR_SCALE, n_draw=20000)
print(f"under the primary prior (scale {PRIOR_SCALE}):")
print(f"  P(beta_bar < 0)                 = {np.mean(sim['beta_bar'] < 0):.3f}   <- 0.5 means symmetric: "
      f"the prior does not favour the hypothesis")
print(f"  P(>=20% attenuation) a priori   = {np.mean(sim['beta_bar'] < ATTENUATION_20):.3f}")
print(f"  P(>=50% attenuation) a priori   = {np.mean(sim['beta_bar'] < np.log(0.5)):.3f}   <- reachable, "
      f"so a real strong effect will not be shrunk away")
print(f"  central 95% of the ratio        = [{np.exp(np.quantile(sim['beta_bar'], .025)):.2f}, "
      f"{np.exp(np.quantile(sim['beta_bar'], .975)):.2f}]")

### The H2 phase term

Model C cuts the phase circle into eight equal slices and gives each its own offset. The estimand
$\sigma_\phi$ is the spread of those offsets: how much the lock deficit moves as the signal slides
through its cycle. Its prior, $\text{Half-}\mathcal N(0,0.25)$, puts mass on both sides of the
equivalence boundary $\log 1.1$, so the H2 verdict is decided by the data rather than by the prior.

In [ ]:
sigma_phase_prior = np.abs(RNG.normal(0, 0.25, 40000))     # Half-Normal(0.25)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].hist(sigma_phase_prior, bins=90, density=True, color="steelblue", alpha=.8)
ax[0].axvline(ROPE_LOG, color="crimson", ls="--", label=f"ROPE edge log 1.1 = {ROPE_LOG:.3f}")
ax[0].set_title(r"prior on the phase spread $\sigma_\phi$"); ax[0].legend(fontsize=8)
ax[0].set_xlabel(r"$\sigma_\phi$")

# what a prior-plausible set of per-phase offsets looks like
for _ in range(25):
    ax[1].plot(np.arange(N_PHASE_BINS := 8),
               RNG.choice(sigma_phase_prior) * RNG.normal(size=8),
               marker="o", ms=3, alpha=.35, lw=1, color="steelblue")
ax[1].axhline(0, color="k", lw=.8)
ax[1].set_xlabel("phase bin"); ax[1].set_ylabel("offset")
ax[1].set_title("prior-plausible per-phase offsets")
fig.tight_layout(); fig.savefig(FIG_DIR / "P1_prior_predictive_phase.png", dpi=140,
                                bbox_inches="tight"); plt.show()

print(f"P(sigma_phase < ROPE) a priori = {np.mean(sigma_phase_prior < ROPE_LOG):.3f}   "
      f"<- mass on both sides, so the H2 verdict comes from the data, not this prior")

## 1.3, Prior predictive for Model B (H1 representational)

The response is a prequential codelength in bits: strictly positive, right-skewed, and with a
variance that grows with its mean. That is what the Gamma likelihood with a log link is for, and it
is why $\theta_{lock}$ is read multiplicatively, $e^{\theta_{lock}}$ is the factor by which
describing the labels costs more at a locked frequency.

In [ ]:
n = 20000
# alpha_0 ~ Normal(log(y_bar), 1) is evaluated in units of the observed baseline y_bar.
alpha_offset = RNG.normal(0.0, 1.0, n)
theta_p = RNG.normal(0.0, 0.5, n)
k_p = RNG.gamma(2.0, 1 / 0.1, n)
mu_locked_relative = np.exp(alpha_offset + theta_p)
y_locked_relative = RNG.gamma(k_p, mu_locked_relative / k_p)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].hist(np.exp(theta_p), bins=90, range=(0, 4), density=True, color="darkorange", alpha=.85)
ax[0].axvline(1, color="k", lw=.8)
ax[0].axvline(1.2, color="crimson", ls="--", label="+20% codelength")
ax[0].set_title(r"implied codelength ratio $e^{\theta_{lock}}$")
ax[0].legend(fontsize=8)
ax[1].hist(np.clip(y_locked_relative, 0, 20), bins=90, density=True,
           color="darkorange", alpha=.85)
ax[1].set_title(r"prior predictive locked codelength $L/\bar L$")
fig.tight_layout()
fig.savefig(FIG_DIR / "P1_prior_predictive_codelength.png", dpi=140, bbox_inches="tight")
plt.show()

print(f"P(theta_lock > 0) a priori = {np.mean(theta_p > 0):.3f}")
print(f"P(codelength expands by >=20%) = {np.mean(theta_p > np.log(1.2)):.3f}")
print("The absolute bit scale is supplied by each observed table through log(mean(L)); "
      "the prior predictive therefore does not invent a 40-bit baseline.")

## 1.4, Prior predictive for Models D1 and D2 (H3)

D1 asks whether the token dispersion is lower on a predicted grid than off it: $\theta_S$ is that
difference, on a log scale, and H3 predicts $\theta_S<0$. Its $\mathcal N(0,1)$ prior is symmetric,
so a dip is not assumed.

D2 asks whether the measured comb spacing follows the predicted one. It is a plain regression of
measured spacing on $f_s/S$ and $f_s/P$, and **H3 predicts a slope of exactly 1 on $f_s/S$ and 0 on
$f_s/P$**. The prior on both slopes is centred on $0$, *no* relationship, so the hypothesis has to
be earned from the data.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))

theta_S = RNG.normal(0, 1, 20000)
ax[0].hist(theta_S, bins=80, density=True, color="steelblue", alpha=.85)
ax[0].axvline(0, color="k", lw=.9)
ax[0].set_title(r"D1 prior on $\theta_S$ (H3 predicts $<0$)")

kappa_S = RNG.normal(0, 1, 20000)
ax[1].hist(kappa_S, bins=80, density=True, color="seagreen", alpha=.85)
ax[1].axvline(1.0, color="crimson", ls="--", label=r"$\kappa_F=1$")
ax[1].axvspan(1 - ROPE_SLOPE, 1 + ROPE_SLOPE, color="crimson", alpha=.12)
ax[1].axvline(0.0, color="k", lw=.8)
ax[1].set_title(r"D2 prior on $\kappa_F$")
ax[1].legend(fontsize=8)

predicted = RNG.choice([pl.FS / S for _, S in FULL_MODELS]
                       + [pl.FS / P for P, _ in FULL_MODELS], size=20000)
sigma_extra = np.abs(RNG.normal(0, 5, 20000))
f1_prior = RNG.normal(kappa_S * predicted, np.sqrt(sigma_extra ** 2 + 1.0 ** 2))
ax[2].hist(np.clip(f1_prior, -100, 250), bins=100, density=True, color="mediumpurple", alpha=.8)
ax[2].axvspan(*pl.BAND, color="0.5", alpha=.08, label="analysed band")
ax[2].set_title(r"D2 prior predictive $\hat f_1$ [Hz]")
ax[2].legend(fontsize=8)

fig.tight_layout()
fig.savefig(FIG_DIR / "P1_prior_predictive_H3.png", dpi=140, bbox_inches="tight")
plt.show()
print(f"P(kappa_F in ROPE around 1) a priori = "
      f"{np.mean(np.abs(kappa_S - 1) < ROPE_SLOPE):.3f}")
print(f"P(prior-predictive f1 < 0) = {np.mean(f1_prior < 0):.3f}; this exposes the deliberately "
      "broad Normal D2 prior fixed in Deliverable 3 rather than hiding it.")

## 1.5, Checkpoint

The prior specification and its predictive summaries are stored so the reported analysis can be
traced back to priors that were fixed before the data existed.

In [ ]:
prior_summary = {
    "prior_scale": PRIOR_SCALE, "prior_ladder": PRIOR_LADDER, "nu": NU,
    "attenuation_20": float(ATTENUATION_20), "rope_log": float(ROPE_LOG),
    "rope_slope": ROPE_SLOPE,
    "p_attenuation20_prior": float(np.mean(sim["beta_bar"] < ATTENUATION_20)),
    "p_sigma_phase_in_rope_prior": float(np.mean(sigma_phase_prior < ROPE_LOG)),
    "p_kappaS_in_rope_prior": float(np.mean(np.abs(kappa_S - 1) < ROPE_SLOPE)),
    "support_skeleton_rows": int(len(SKELETON)),
    "full_contrast_rows": int(FULL_CONTRAST_ROWS), "seed": SEED,
}
save_json(prior_summary, "01_prior_spec.json")
save_df(PRIOR_SPEC, "01_prior_spec.parquet")
print(json.dumps(prior_summary, indent=2))

---
# Part 2, Observation: Chronos produces the data

This is the only part that loads a model, and the only part that wants a GPU. It runs
`collect.py`, which visits the 15 geometries and writes five tidy tables. Everything after this
point treats those tables as plain data.

**What is measured, and why each measurement exists**

| table | measurement | why |
|---|---|---|
| `contrasts` | forecast amplitude recovery $R$ at each lock $f_k$ and at both controls $f_k\pm 0.25 f_s/S$, sharing background and phase | the *behavioural* endpoint of Eq. (8); the phase index is kept so H2 is testable |
| `mdl_cells` | prequential codelength $L(D)$ of a probe separating $f_c-1$ Hz from $f_c+1$ Hz, per probe stage | the *representational* endpoint of Eq. (10) |
| `mdl_bandtasks` | the seven hierarchical band tasks, with shuffled-label and random-init controls | descriptive cross-check of overall decodability |
| `collapse` | across-patch token dispersion $z_g(f)$ on the union grid, in three signal modes | the location endpoint of Eq. (12) |
| `sites` | detected collapse sites, their fundamental $f_1$ and spacing $\hat\Delta$ | the movement endpoint of Eq. (13) |

**Signals.** Following the deliverable's Data section, the tone rides on a unit-variance
**TSMixup** or **KernelSynth** background at SNR 4, the two synthetic corpora the project
generates. The collapse sweep additionally records the **pure sinusoid** mode, because only there
is the degeneracy exact ($z=0$ when consecutive patches coincide); the background modes show the
same comb as a deep dip and are the realistic-signal cross-check.

**Resumption.** `collect.py` shards per geometry under `data/raw/`. A geometry whose shards exist
is skipped *and its model is never loaded*, so an interrupted run costs only the geometry it died
on.

In [ ]:
banner("PART 2 - OBSERVATION (CHRONOS)")

import collect

CFG = collect.Config()
CFG.batch_size = 64
assert list(pl.DELIVERABLE3_MODELS) == FULL_MODELS
assert not CFG.smoke

print("reportable design:", {k: v for k, v in vars(CFG).items() if k != "generators"})
print("generators:", CFG.generators)
print("session:", [pl.model_tag(P, S) for P, S in SESSION_MODELS])

## 2.1 - Collection and MCMC cost before execution

In [ ]:
# ---- what Part 2 costs, before it starts ---------------------------------------------------
# On a CPU runtime the collection is the long pole, so this cell counts the forward passes each
# geometry needs and turns them into an estimate. Nothing here loads a model unless CALIBRATE is
# on, and the counts are exact: they are read from the same probe_lib functions the collectors
# loop over.
BUDGET_HOURS = 70.0     # what you have to spend, for the session split printed at the end
CALIBRATE    = True     # time one batch on the smallest geometry to get real seconds per pass
SEC_FORECAST = 0.035    # fallback, seconds per forecast on one CPU core-set; overwritten if
SEC_CAPTURE  = 0.020    # CALIBRATE succeeds. A forecast decodes the horizon, a capture does not.


def _counts(P, S, cfg):
    """Exact forward-pass counts for one geometry, split into forecasts and state captures."""
    offsets = {f: pl.control_offset(P, S, f) for f in pl.f_lock(P, S)}
    sites = [f for f, d in offsets.items() if np.isfinite(d)]
    n_gen = len(cfg.generators)

    # contrasts: every triplet arm, at every phase, on every background
    fore = sum(len(pl.phases_Sf(fk, cfg.n_phase_contrast)) for fk in sites) * cfg.n_bg * 3 * n_gen

    # mdl cells: two classes per centre, capped at mdl_n_per_class contexts each
    centers = {round(f, 6) for fk in sites
               for f in (fk, fk - offsets[fk], fk + offsets[fk])}
    cap = len(centers) * 2 * cfg.mdl_n_per_class

    # band tasks: the descriptive sweep, plus the untrained clone on a 4x coarser grid
    if cfg.band_tasks:
        fl = np.arange(pl.BAND[0], pl.BAND[1] + 1e-9, cfg.bt_step)
        cap += sum(len(pl.phases_Sf(f, cfg.bt_n_phase)) for f in fl)
        if cfg.bt_random_init:
            co = np.arange(pl.BAND[0], pl.BAND[1] + 1e-9, cfg.bt_step * 4)
            cap += sum(len(pl.phases_Sf(f, max(2, cfg.bt_n_phase // 2))) for f in co)

    # collapse: the union grid, once per replicate per signal mode
    cap += len(pl.union_grid(FULL_MODELS, cfg.collapse_step)) * cfg.collapse_reps * \
        len(cfg.collapse_modes)
    return fore, cap


if CALIBRATE and HAVE_TORCH:
    try:
        import time
        _p = pl.Probe(8, 8, device=CFG.device, batch_size=CFG.batch_size)
        try:
            _n = min(32, CFG.batch_size)
            _ctx = np.stack([pl.build_context(None, 40.0, 0.0, pl.CTX + pl.PRED)] * _n)
            _t = time.perf_counter()
            _p.recovery(_ctx[:, :pl.CTX], _ctx[:, pl.CTX:], np.full(_n, 40.0))
            SEC_FORECAST = (time.perf_counter() - _t) / _n
            _t = time.perf_counter()
            _p.capture_reg(_ctx[:, :pl.CTX])
            SEC_CAPTURE = (time.perf_counter() - _t) / _n
        finally:
            _p.close()
        print(f"calibrated on p8-s8, device={_p.device}: {SEC_FORECAST*1000:.1f} ms per forecast, "
              f"{SEC_CAPTURE*1000:.1f} ms per capture")
    except Exception as _e:
        print(f"calibration skipped ({type(_e).__name__}: {_e}); using the defaults above")
else:
    print("calibration off or torch unavailable; using the defaults above")

rows = []
for _P, _S in FULL_MODELS:
    f, c = _counts(_P, _S, CFG)
    rows.append(dict(model=pl.model_tag(_P, _S), forecasts=f, captures=c,
                     hours=(f * SEC_FORECAST + c * SEC_CAPTURE) / 3600))
COST = pd.DataFrame(rows).sort_values("hours")
COST["cumulative_h"] = COST["hours"].cumsum()
display(COST.round({"hours": 2, "cumulative_h": 2}))

_tot = COST["hours"].sum()
print(f"\ntotal: {COST.forecasts.sum():,} forecasts + {COST.captures.sum():,} captures "
      f"= {_tot:.1f} h at the rates above")
print(f"budget: {BUDGET_HOURS:.0f} h  ->  "
      + ("fits, with %.0f h to spare" % (BUDGET_HOURS - _tot) if _tot <= BUDGET_HOURS
         else "SHORT by %.0f h; see the note below" % (_tot - BUDGET_HOURS)))

# A session plan. Shards are written per table per geometry and skipped on the next run, so a
# session that ends mid-geometry loses only the table in flight.
SESSION_H = 3.0
_grp, _acc, _plan = [], 0.0, []
for _, r in COST.iterrows():
    if _acc + r.hours > SESSION_H and _grp:
        _plan.append((_grp, _acc)); _grp, _acc = [], 0.0
    _grp.append(r.model); _acc += r.hours
if _grp:
    _plan.append((_grp, _acc))
print(f"\nsuggested split into ~{SESSION_H:g} h sessions:")
for _i, (_g, _h) in enumerate(_plan, 1):
    print(f"  session {_i} ({_h:.1f} h):  models={_g}")
print("\nrun one session with, for example:")
print("    DATA = collect.collect_all(DATA_DIR, models=[(8, 8), (16, 8)], cfg=CFG, planned_models=FULL_MODELS, allow_partial=True)")
if not HAVE_TORCH:
    print("\ntorch is not available here, so Part 2 cannot run in this runtime at all.")

print("\nThe estimate above covers Chronos forward passes only. It excludes parameter recovery, "
      "the empirical posterior fits, four full sensitivity refits, LOO and posterior-predictive "
      "simulation. Those stages use 4 chains and the preregistered 2000 tune + 2000 retained "
      "draws; do not interpret this collection estimate as total wall time.")

In [ ]:
# Session subsets are allowed only as resumable collection work. Inference remains locked until
# the manifest contains complete, validated shards for all 15 planned geometries.
if HAVE_TORCH:
    DATA = collect.collect_all(
        DATA_DIR,
        models=SESSION_MODELS,
        cfg=CFG,
        planned_models=FULL_MODELS,
        allow_partial=set(SESSION_MODELS) != set(FULL_MODELS),
    )
    collection_manifest = json.loads((DATA_DIR / collect.MANIFEST_NAME).read_text(encoding="utf-8"))
    if collection_manifest["status"] != "complete":
        completed = sorted({key.split("__", 1)[1] for key in collection_manifest["shards"]})
        raise RuntimeError(
            "This collection session is safely checkpointed but the 15-model design is partial. "
            f"Completed shard tags: {completed}. Run the remaining sessions; do not continue to MCMC.")
else:
    print("Torch/Chronos unavailable: validating a completed Drive collection.")

DATA = collect.load_collection(DATA_DIR, cfg=CFG, planned_models=FULL_MODELS, require_complete=True)
collection_manifest_path = DATA_DIR / collect.MANIFEST_NAME
ANALYSIS_MANIFEST["collection_manifest_sha256"] = cp.sha256_file(collection_manifest_path)
cp.atomic_json(ANALYSIS_MANIFEST_PATH, ANALYSIS_MANIFEST)

contrasts = DATA["contrasts"]
mdl_cells = DATA["mdl_cells"]
collapse = DATA["collapse"]
sites = DATA["sites"]
bandtasks = DATA.get("mdl_bandtasks")
print("rows:", {name: len(frame) for name, frame in DATA.items()})

## 2.2 - Generated-signal archive and quality gate

In [ ]:
sig_dir = DATA_DIR / "signals"
sig_index = pd.read_parquet(sig_dir / "signals_index.parquet")
required_signal_columns = {
    "generator", "seed", "mean", "std", "sha256", "recipe_sha256",
    "pool_max_abs_corr", "pool_effective_rank", "pool_quality_ok",
}
if not required_signal_columns.issubset(sig_index.columns):
    raise ValueError("signal archive predates the Deliverable 3 quality/provenance schema")
if not sig_index["pool_quality_ok"].all():
    raise ValueError("a generated-signal pool failed its quality gate")
for row in sig_index.itertuples():
    if cp.sha256_file(sig_dir / row.file) != row.sha256:
        raise ValueError(f"signal hash mismatch: {row.file}")

display(sig_index.groupby("generator").agg(
    draws=("file", "size"), max_abs_mean=("mean", lambda x: float(np.max(np.abs(x)))),
    mean_std=("std", "mean"), max_abs_corr=("pool_max_abs_corr", "first"),
    effective_rank=("pool_effective_rank", "first"),
))

fig, axes = plt.subplots(2, 2, figsize=(13, 5), sharex=True)
for row_index, generator in enumerate(CFG.generators):
    background = np.load(sig_dir / f"background_{generator}_bg0.npy", allow_pickle=False)
    time = np.arange(len(background)) / pl.FS
    axes[row_index][0].plot(time, background, lw=.8, color="0.4")
    axes[row_index][0].axvline(pl.CTX / pl.FS, color="k", ls=":", lw=1)
    axes[row_index][0].set_ylabel(generator)
    model_input = pl.build_context(background, 64.0, 0.0, len(background))
    axes[row_index][1].plot(time, model_input, lw=.8, color="#c62828")
axes[0][0].set_title("centred, unit-variance archived background")
axes[0][1].set_title("background + 64 Hz tone at SNR 4")
fig.tight_layout()
fig.savefig(FIG_DIR / "P2_signals.png", dpi=140, bbox_inches="tight")
plt.show()

## 2.3 - Does the collected design support the inference?

In [ ]:
design_check = collect.check_design(DATA, expected_models=FULL_MODELS, cfg=CFG)
display(pd.DataFrame(design_check["d2_identification"]).T)

live = contrasts[contrasts["live"].astype(bool)].reset_index(drop=True)
if live.empty:
    raise ValueError("no live local contrasts remain after the preregistered recovery floor")

balance = contrasts.groupby(["model", "P", "S"]).agg(
    rows=("d", "size"), live=("live", "sum"), locks=("f_lock", "nunique"),
    phases=("phase_idx", "nunique"), generators=("generator", "nunique"),
    backgrounds=("bg_id", "nunique"),
).reset_index()
display(balance)
print(f"live contrasts entering A/C: {len(live):,}/{len(contrasts):,}")

## 2.4 - Raw observations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))

# (a) the contrast d against cycles-per-patch, live rows only. d<0 is the localized loss H1 predicts.
live = contrasts[contrasts["live"]].reset_index(drop=True)
for model, g in live.groupby("model"):
    axes[0].scatter(g["cpp"], g["d"], s=14, alpha=.55, label=model)
axes[0].axhline(0, color="k", lw=.9)
axes[0].set_xlabel("cycles per patch  cpp = $f_k P / f_s$")
axes[0].set_ylabel("$d$   (negative = attenuation at the lock)")
axes[0].set_title("Eq. (8) local contrast, live locks only"); axes[0].legend(fontsize=7)

# (b) the collapse profile of one geometry, with both competing grids drawn on it
sub = collapse[(collapse["model"] == "p16-s8") & (collapse["mode"] == "tsmixup")]
if len(sub) == 0:
    sub = collapse[collapse["model"] == collapse["model"].iloc[0]]
prof = sub.groupby("f", as_index=False)["z"].mean().sort_values("f")
P0, S0 = int(sub["P"].iloc[0]), int(sub["S"].iloc[0])
axes[1].plot(prof["f"], prof["z"], color="#c62828", lw=1.3)
for j, f in enumerate(pl.stride_locks(S0)):
    axes[1].axvline(f, color="#6a1b9a", ls="--", lw=1, alpha=.75,
                    label=f"predicted stride lock $c f_s/S$ (S={S0})" if j == 0 else None)
for j, f in enumerate(pl.patch_nulls(P0)):
    axes[1].axvline(f, color="#1565c0", ls=":", lw=1, alpha=.6,
                    label=f"predicted patch null $k f_s/P$ (P={P0})" if j == 0 else None)
axes[1].set_xlabel("frequency [Hz]"); axes[1].set_ylabel("across-patch token dispersion $z$")
axes[1].set_title(f"Eq. (12) collapse profile, p{P0}-s{S0}"); axes[1].legend(fontsize=7)

fig.tight_layout(); fig.savefig(FIG_DIR / "P2_raw_data.png", dpi=140, bbox_inches="tight"); plt.show()

display(sites[sites.rep == -1][["model", "P", "S", "mode", "n_sites", "f1", "delta_hat"]]
        .assign(fs_over_S=lambda d: (pl.FS / d["S"]).round(2),
                fs_over_P=lambda d: (pl.FS / d["P"]).round(2)).round(2))

---
# Part 3, The likelihood

Part 1 fixed the priors; Part 2 produced the observations. This part supplies the third ingredient
and, more importantly, **checks that it works before it is trusted**.

Three things happen here:

1. **Ãƒâ€šSection 3.1, the likelihood, written out.** The Student-$t_4$ density that Eq. (9) prescribes is
   implemented directly in NumPy and checked against SciPy. Seeing it as an explicit function makes
   concrete what the sampler is otherwise doing invisibly, and shows *why* $t_4$ rather than a
   Gaussian: its tails make a handful of extreme contrasts cost far less, so the population effect
   is not dragged around by them.
2. **Ãƒâ€šSection 3.2, the five models.** Each is a small factory function returning a PyMC model.
3. **Ãƒâ€šSection 3.3, parameter recovery.** The likelihood is used to *simulate* contrasts with a known
   effect on the real design matrix, and the model is refitted. If a known $\log 0.5$ cannot be
   recovered from this design, no posterior fitted on the real data means anything.

## 3.1, The Student-$t_4$ contrast likelihood

For observation $i$ with linear predictor $\mu_i$ and scale $\sigma$,

$$\log p(d_i \mid \mu_i, \sigma, \nu) = \log\Gamma\!\Big(\tfrac{\nu+1}{2}\Big) - \log\Gamma\!\Big(\tfrac{\nu}{2}\Big) - \tfrac12\log(\pi\nu) - \log\sigma - \tfrac{\nu+1}{2}\log\!\Big(1 + \tfrac{(d_i-\mu_i)^2}{\nu\sigma^2}\Big).$$

The last term is the whole point. A Gaussian charges a residual quadratically, so one contrast ten
scale units away costs 50 units of log-likelihood and the fit will contort itself to reduce it. The
$t_4$ charges it logarithmically, about 5 units, so an outlier is *tolerated* rather than
obeyed. With recovery ratios that can collapse to ~0 at a dead frequency, that robustness is not a
stylistic preference; it is what keeps a handful of dead cells from setting $\bar\beta$.

In [ ]:
banner("PART 3, LIKELIHOOD")

from scipy.special import gammaln

def student_t_logpdf(d, mu, sigma, nu=NU):
    """Log density of Student-t(nu, mu, sigma), written out rather than imported."""
    z2 = ((np.asarray(d) - mu) / sigma) ** 2
    return (gammaln((nu + 1) / 2) - gammaln(nu / 2) - 0.5 * np.log(np.pi * nu)
            - np.log(sigma) - (nu + 1) / 2 * np.log1p(z2 / nu))

# cross-check against SciPy: the two must agree to machine precision
_d = np.linspace(-6, 6, 501)
assert np.allclose(student_t_logpdf(_d, 0.3, 0.7), stats.t.logpdf(_d, NU, loc=0.3, scale=0.7))
print("student_t_logpdf matches scipy.stats.t.logpdf to machine precision")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(_d, np.exp(student_t_logpdf(_d, 0, 1)), label=r"Student-$t_4$", lw=1.8)
ax[0].plot(_d, stats.norm.pdf(_d), label="Normal", lw=1.4, ls="--")
ax[0].set_title("density"); ax[0].legend(fontsize=8)
ax[1].plot(_d, -student_t_logpdf(_d, 0, 1), label=r"Student-$t_4$", lw=1.8)
ax[1].plot(_d, -stats.norm.logpdf(_d), label="Normal", lw=1.4, ls="--")
ax[1].set_title("cost of a residual  $-\\log p$   (why $t_4$: outliers are tolerated, not obeyed)")
ax[1].legend(fontsize=8); ax[1].set_xlabel("residual / scale")
fig.tight_layout(); fig.savefig(FIG_DIR / "P3_likelihood.png", dpi=140, bbox_inches="tight"); plt.show()

### The likelihood is what makes the effect identifiable

A quick profile makes the point that Part 1's prior predictive could not: holding the nuisance
parameters at sensible values and sweeping $\bar\beta$, the observed contrasts pick out a narrow
range. The prior was flat over a factor of ~3; the likelihood is not.

In [ ]:
d_live = live["d"].to_numpy()
grid_b = np.linspace(-2.5, 2.5, 400)
ll = np.array([student_t_logpdf(d_live, b, np.std(d_live)).sum() for b in grid_b])
prior_ll = stats.t.logpdf(grid_b, NU, scale=PRIOR_SCALE)

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(grid_b, ll - ll.max(), label="log-likelihood profile (data)", lw=1.8)
ax.plot(grid_b, prior_ll - prior_ll.max(), label="log prior", lw=1.4, ls="--")
ax.axvline(grid_b[ll.argmax()], color="crimson", lw=1,
           label=f"profile maximum = {grid_b[ll.argmax()]:+.3f}")
ax.set_xlabel(r"$\bar\beta$"); ax.set_ylabel("relative log density"); ax.set_ylim(-30, 1)
ax.legend(fontsize=8); ax.set_title("the data, not the prior, locate the effect")
fig.tight_layout(); fig.savefig(FIG_DIR / "P3_profile.png", dpi=140, bbox_inches="tight"); plt.show()

## 3.2 - The five Deliverable 3 model families

In [ ]:
def _codes(series):
    """Integer codes plus the ordered level names, for PyMC `coords`."""
    codes, levels = pd.factorize(series)
    return np.asarray(codes), list(map(str, levels))


def _overlap_scaled(df: pd.DataFrame, cfg_levels: list[str]) -> np.ndarray:
    """Centred, scaled patch overlap O = (P-S)/P; one unit = 0.5 of overlap."""
    O = df.groupby("model")["overlap"].first().reindex(cfg_levels).to_numpy(float)
    return (O - O.mean()) / 0.5


def _logP_centred(df: pd.DataFrame, cfg_levels: list[str]) -> np.ndarray:
    """Centred log patch size.

    Deliverable 3, H1: "The overlap enters as a ratio and the patch size as log P, because the
    patch grid has spacing fs/P: equal steps in log P are then equal ratios of spacing." On a raw-P
    scale one coefficient would make 8->16 and 16->24 the same change, which the geometry does not.
    """
    Pv = df.groupby("model")["P"].first().reindex(cfg_levels).to_numpy(float)
    lp = np.log(Pv)
    return lp - lp.mean()


# --------------------------------------------------------------------------------------- #
def model_A_contrast(df: pd.DataFrame, scale: float = PRIOR_SCALE, nu: int = NU,
                     likelihood: str = "student", config_level: str = "both") -> pm.Model:
    """Model A, H1 behavioural, and the configuration level that carries M1.

        d_i ~ Student-t(nu, mu_i, sigma)
        mu_i = beta[config_i] + u_harm[i] + u_bg[i]
        beta_c ~ Normal(beta_bar + delta_O * Otilde_c + delta_P * logPtilde_c, tau)

    `config_level` selects which covariates the configuration level carries, which is how M1 is
    decided: Deliverable 3 asks "whether what varies between geometries is the overlap ratio,
    the absolute stride, or the patch size", and the four fits below are compared by LOO.

        "both"     beta_bar + delta_O * Otilde + delta_P * logPtilde   (the reported model)
        "overlap"  beta_bar + delta_O * Otilde
        "patch"    beta_bar + delta_P * logPtilde
        "none"     beta_bar

    `likelihood="normal"` swaps in a Gaussian for the Part 5 robustness check.
    """
    cfg_c, cfg_l = _codes(df["model"])
    harm_c, harm_l = _codes(df["f_lock"].round(3).astype(str))
    bg_c, bg_l = _codes(df["generator"] + "#" + df["bg_id"].astype(str))
    O_t = _overlap_scaled(df, cfg_l)
    lP_t = _logP_centred(df, cfg_l)
    y = df["d"].to_numpy(float)

    coords = {"config": cfg_l, "harmonic": harm_l, "background": bg_l, "obs": np.arange(len(y))}
    with pm.Model(coords=coords) as m:
        #, population level: the estimand of H1 -------------------------------------
        beta_bar = pm.StudentT("beta_bar", nu=nu, mu=0.0, sigma=scale)
        cfg_mean = beta_bar
        if config_level in ("both", "overlap"):
            delta_O = pm.StudentT("delta_O", nu=nu, mu=0.0, sigma=scale)   # M1
            cfg_mean = cfg_mean + delta_O * O_t
        if config_level in ("both", "patch"):
            delta_P = pm.StudentT("delta_P", nu=nu, mu=0.0, sigma=scale)
            cfg_mean = cfg_mean + delta_P * lP_t

        #, hierarchy: configuration, lock harmonic, background realisation ----------
        tau = pm.HalfStudentT("tau", nu=nu, sigma=scale)
        z_cfg = pm.Normal("z_cfg", 0.0, 1.0, dims="config")          # non-centred
        beta = pm.Deterministic("beta", cfg_mean + tau * z_cfg, dims="config")

        sigma_h = pm.HalfStudentT("sigma_harm", nu=nu, sigma=scale)
        u_h = pm.Deterministic("u_harm", sigma_h * pm.Normal("z_harm", 0.0, 1.0, dims="harmonic"),
                               dims="harmonic")
        sigma_b = pm.HalfStudentT("sigma_bg", nu=nu, sigma=scale)
        u_b = pm.Deterministic("u_bg", sigma_b * pm.Normal("z_bg", 0.0, 1.0, dims="background"),
                               dims="background")

        mu = beta[cfg_c] + u_h[harm_c] + u_b[bg_c]
        sigma = pm.HalfStudentT("sigma", nu=nu, sigma=scale)
        if likelihood == "normal":
            pm.Normal("d", mu=mu, sigma=sigma, observed=y, dims="obs")
        else:
            pm.StudentT("d", nu=nu, mu=mu, sigma=sigma, observed=y, dims="obs")

        # reported on the natural scale: the multiplicative change in recovery at a lock
        pm.Deterministic("recovery_ratio", pm.math.exp(beta_bar))
    return m


# --------------------------------------------------------------------------------------- #
def model_B_codelength(df: pd.DataFrame, adjusted: bool = True) -> pm.Model:
    """Model B, H1 representational.  Deliverable Eq. (10).

        y_i ~ Gamma(shape=k, mean=mu_i),   log mu_i = alpha_0 + theta_lock * IsLocked

    `adjusted=True` adds zero-mean stage and geometry offsets. Eq. (10) taken literally pools
    codelengths from probe stages whose scales differ several-fold, which inflates the residual
    without touching the estimand; blocking on stage is the standard remedy and leaves theta_lock
    exactly as specified. The two versions are compared by LOO in Part 4.
    """
    y = np.clip(df["L_bits"].to_numpy(float), 1e-3, None)    # Gamma support is strictly positive
    locked = df["is_locked"].to_numpy(float)
    st_c, st_l = _codes(df["stage"])
    mo_c, mo_l = _codes(df["model"])

    coords = {"stage": st_l, "geometry": mo_l, "obs": np.arange(len(y))}
    with pm.Model(coords=coords) as m:
        alpha0 = pm.Normal("alpha_0", mu=float(np.log(np.mean(y))), sigma=1.0)
        theta = pm.Normal("theta_lock", mu=0.0, sigma=0.5)       # the estimand
        k = pm.Gamma("k", alpha=2.0, beta=0.1)                   # Gamma shape (dispersion)

        log_mu = alpha0 + theta * locked
        if adjusted:
            s_st = pm.HalfNormal("sigma_stage", 1.0)
            s_mo = pm.HalfNormal("sigma_geometry", 1.0)
            eff_st = pm.Deterministic("eff_stage", s_st * pm.Normal("z_stage", 0, 1, dims="stage"),
                                      dims="stage")
            eff_mo = pm.Deterministic("eff_geometry",
                                      s_mo * pm.Normal("z_geometry", 0, 1, dims="geometry"),
                                      dims="geometry")
            log_mu = log_mu + eff_st[st_c] + eff_mo[mo_c]

        mu = pm.math.exp(log_mu)
        pm.Gamma("y", alpha=k, beta=k / mu, observed=y, dims="obs")
        pm.Deterministic("codelength_ratio", pm.math.exp(theta))
    return m


# --------------------------------------------------------------------------------------- #
N_PHASE_BINS = 8       # how many equal slices of the phase circle get their own offset

def model_C_phase(df: pd.DataFrame, with_phase: bool = True, scale: float = PRIOR_SCALE,
                  nu: int = NU) -> pm.Model:
    """Model C, H2.  Does the lock deficit depend on WHERE in its cycle the signal starts?

        y_i = log(R_ctrl + .01) - log(R_lock + .01)          (the deficit; = -d)
        E[y_i] = beta[config] + u_harm + u_phase[bin of phi_i]
        u_phase ~ Normal(0, sigma_phase)

    The phase circle is cut into N_PHASE_BINS equal slices and each slice gets its own offset.
    `sigma_phase` is then the spread of those offsets: how much the deficit moves as the signal
    slides through its cycle. H2 says the deficit is a property of the stride/patch geometry, not
    of the signal's phase, i.e. sigma_phase ~ 0.

    This replaced a first-harmonic (a cos phi + b sin phi) formulation. The variance component is
    easier to state and to defend, it is the direct Bayesian counterpart of the phase spread the
    project's frequentist suite already reports, and it is strictly more general: it catches ANY
    dependence on phase, not only a sinusoidal one.

    `with_phase=False` builds the nested null with no phase term, for the LOO comparison.
    """
    cfg_c, cfg_l = _codes(df["model"])
    harm_c, harm_l = _codes(df["f_lock"].round(3).astype(str))
    # the background realisation (generator + draw) is a grouping factor here exactly as in Model A:
    # without it, variation between TSMixup and KernelSynth backgrounds would be pushed into the
    # residual and could inflate or mask the phase term this model exists to measure
    bg_c, bg_l = _codes(df["generator"] + "#" + df["bg_id"].astype(str))
    # phase matters only modulo 2*pi: it is the tone's alignment with the patch grid
    phi = np.mod(df["phase"].to_numpy(float), 2 * np.pi)
    bin_c = np.floor(phi / (2 * np.pi / N_PHASE_BINS)).astype(int)
    bin_l = [f"{j * 360 // N_PHASE_BINS}-{(j + 1) * 360 // N_PHASE_BINS} deg" for j in range(N_PHASE_BINS)]
    y = df["y_deficit"].to_numpy(float)

    coords = {"config": cfg_l, "harmonic": harm_l, "background": bg_l, "phasebin": bin_l,
              "obs": np.arange(len(y))}
    with pm.Model(coords=coords) as m:
        beta_bar = pm.StudentT("beta_bar", nu=nu, mu=0.0, sigma=scale)
        tau = pm.HalfStudentT("tau", nu=nu, sigma=scale)
        beta = pm.Deterministic("beta", beta_bar + tau * pm.Normal("z_cfg", 0, 1, dims="config"),
                                dims="config")
        sigma_h = pm.HalfStudentT("sigma_harm", nu=nu, sigma=scale)
        u_h = pm.Deterministic("u_harm", sigma_h * pm.Normal("z_harm", 0, 1, dims="harmonic"),
                               dims="harmonic")
        sigma_b = pm.HalfStudentT("sigma_bg", nu=nu, sigma=scale)
        u_b = pm.Deterministic("u_bg", sigma_b * pm.Normal("z_bg", 0, 1, dims="background"),
                               dims="background")

        mu = beta[cfg_c] + u_h[harm_c] + u_b[bg_c]
        if with_phase:
            # the estimand: the spread of the per-phase offsets
            sigma_phase = pm.HalfNormal("sigma_phase", 0.25)
            u_p = pm.Deterministic("u_phase",
                                   sigma_phase * pm.Normal("z_phase", 0, 1, dims="phasebin"),
                                   dims="phasebin")
            mu = mu + u_p[bin_c]

        sigma = pm.HalfStudentT("sigma", nu=nu, sigma=scale)
        pm.StudentT("y", nu=nu, mu=mu, sigma=sigma, observed=y, dims="obs")
    return m


# --------------------------------------------------------------------------------------- #
COMB_FLOOR = 1e-2      # z_norm floor before the log; see the note in Ãƒâ€šSection 4.4
GRID_TOL_HZ = 1.0      # how close to a predicted site counts as "on the grid"

def model_D1_sites(df: pd.DataFrame, grid: str) -> pm.Model:
    """Model D1, H3 location.  Deliverable Eq. (12).

        log z_g(f) ~ Normal(alpha_g + theta_S 1[f on stride grid] + theta_P 1[f on patch grid], sigma)

    Each frequency is labelled by which predicted grid it falls on (within GRID_TOL_HZ), and the
    model asks whether the token dispersion is systematically lower there. H3 predicts theta_S < 0.

    `grid` selects the labelling: "both" keeps both indicators (the model H3 states), "stride" only
    c*fs/S, "patch" only k*fs/P, "none" neither. The four are fitted on identical observations and
    compared by LOO,
    which is what identifies WHICH parameter generates the sites, the S<P configurations are the
    ones that separate the two families, since on the P=S diagonal the grids coincide.

    This is deliberately the same shape of model as Eq. (10): a linear predictor on a log scale
    with an indicator variable. It replaced an earlier Gaussian-dip "comb" likelihood with free
    depth and width, which estimated two nuisance quantities nothing downstream used and was far
    harder to justify than the claim it was testing.
    """
    g_c, g_l = _codes(df["model"])
    logz = np.log(np.clip(df["z_norm"].to_numpy(float), COMB_FLOOR, None))

    # membership indicators, built from the same helper that defines the grids everywhere else
    on_stride = np.zeros(len(df))
    on_patch = np.zeros(len(df))
    for gi, _name in enumerate(g_l):
        sel = g_c == gi
        P = int(df.loc[sel, "P"].iloc[0]); S = int(df.loc[sel, "S"].iloc[0])
        f = df.loc[sel, "f"].to_numpy(float)
        on_stride[sel] = (pl.comb_distance(f, pl.FS / S) <= GRID_TOL_HZ).astype(float)
        on_patch[sel] = (pl.comb_distance(f, pl.FS / P) <= GRID_TOL_HZ).astype(float)

    coords = {"geometry": g_l, "obs": np.arange(len(logz))}
    with pm.Model(coords=coords) as m:
        alpha_g = pm.Normal("alpha_g", 0.0, 1.0, dims="geometry")   # off-grid level per geometry
        sigma = pm.HalfNormal("sigma", 1.0)
        mu = alpha_g[g_c]
        if grid in ("stride", "both"):
            theta_S = pm.Normal("theta_S", 0.0, 1.0)
            mu = mu + theta_S * on_stride
        if grid in ("patch", "both"):
            theta_P = pm.Normal("theta_P", 0.0, 1.0)
            mu = mu + theta_P * on_patch
        pm.Normal("logz", mu=mu, sigma=sigma, observed=logz, dims="obs")
    return m


# --------------------------------------------------------------------------------------- #
def model_D2_movement(df: pd.DataFrame, branch: str, resolution_hz: float | None = None,
                      response: str = "f1") -> pm.Model:
    """Models D2, H3a and H3b.  One scaling law per branch, no intercept.

        f1_hat[g] ~ Normal(kappa_F * Delta_F[g], sqrt(sigma_F^2 + df^2))

    Deliverable 3, H3: every detected dip is first assigned to the branch that predicts it and
    ambiguous sites are set aside, then the fundamental of that branch is compared with the spacing
    that branch predicts, Delta_S = fs/S or Delta_P = fs/P. H3a predicts kappa_S = 1 and H3b
    predicts kappa_P = 1: each branch tracks its own parameter one for one.

    This replaced a single regression of one pooled spacing on BOTH predictors, with an intercept
    and the prediction kappa_P = 0. That formulation contradicted H3 as approved in the approved Deliverable 3 formulation,
    which claims that each branch moves with its own parameter; and it treated the detected set as
    one comb, when F_lock is a union and the union of two combs has two interlaced spacings. The
    dip that formulation had to call spurious at p16-s8 is a patch null at fs/P = 32 Hz, i.e. an
    observation H3 predicts.

    **Measurement resolution.** The spacing is read off a frequency sweep, so it carries a floor of
    about one grid step whatever the sampling noise. Stating it is not cosmetic: when the sites land
    on the predicted comb exactly, the residuals vanish, sigma is pushed to zero and the posterior
    develops a funnel NUTS cannot traverse (R-hat above 2, hundreds of divergences). Adding the
    floor in quadrature removes the pathology and is the honest statement: a near-zero sigma is then
    a result, the spacings follow the law to within the sweep resolution, not a sampler failure.

    `response="f1"` uses the branch's fundamental, the lowest detected site of that branch, which is
    what the deliverable specifies. `response="delta_hat"` uses the median gap of the same branch and
    is fitted only as a robustness check: with three or four sites in band one spurious detection
    inserts a short interval and flips the median, to which the fundamental is immune.
    """
    sub = df[df["branch"] == branch]
    y = sub[response].to_numpy(float)                      # measured spacing [Hz]
    x = sub["predicted_spacing"].to_numpy(float)           # fs/S or fs/P [Hz]
    name = {"stride": "kappa_S", "patch": "kappa_P"}[branch]

    step = resolution_hz if resolution_hz is not None else CFG.collapse_step
    with pm.Model(coords={"obs": np.arange(len(y))}) as m:
        kappa = pm.Normal(name, mu=0.0, sigma=1.0)         # H3a / H3b predict 1
        sigma_d = pm.HalfNormal("sigma_extra", 5.0)        # scatter beyond the grid floor [Hz]
        sigma_total = pm.math.sqrt(sigma_d ** 2 + step ** 2)
        pm.Normal("f1_hat", mu=kappa * x, sigma=sigma_total, observed=y, dims="obs")
    return m


def sample(model, name: str, draws=None, tune=None, chains=None, **kw):
    """One place for the sampler settings the deliverable's Inference paragraph prescribes.

    `NUTS_BACKEND` (set in Part 0.3) selects who does the sampling. The model, the priors and the
    number of draws are identical in every case; only the implementation of NUTS changes, so the
    posterior is the same target distribution.

      "pymc"     the default. Pure Python/PyTensor, CPU only.
      "nutpie"   a Rust implementation, several times faster on the same CPU.
      "numpyro"  JAX. This is the one that uses a GPU, and the only reason a GPU runtime helps:
                 PyMC's own sampler never touches it. Chains run vectorised rather than in series.
    """
    extra = dict(kw)
    if NUTS_BACKEND != "pymc":
        extra["nuts_sampler"] = NUTS_BACKEND
        if NUTS_BACKEND == "numpyro":
            extra.setdefault("nuts_sampler_kwargs", {"chain_method": "vectorized"})
    with model:
        return pm.sample(draws=draws or DRAWS, tune=tune or TUNE, chains=chains or CHAINS,
                         cores=1, random_seed=SEED, target_accept=TARGET_ACCEPT,
                         progressbar=False, idata_kwargs={"log_likelihood": True}, **extra)

print("model factories ready: A (H1 behavioural + M1), B (H1 representational), "
      "C (H2), D1 (H3 location), D2 (H3a stride branch, H3b patch branch)")

## 3.3 - Synthetic parameter recovery for every likelihood family

Before empirical fitting, A, B, C, D1 and both D2 branches are exercised on known effects using
the real design axes. Recovery uses four chains and is a fail-closed method gate. These synthetic
numbers are **NON-REPORTABLE** and cannot be cited as evidence about Chronos.

In [ ]:
RECOVERY_VERSION = "d3-all-families-v1"
RECOVERY_DRAWS = 1000
RECOVERY_TUNE = 1000
RECOVERY_FILE = "03_recovery_d3_v1.parquet"

def _recovery_rows(idata, model_name: str, truths: dict[str, float]) -> list[dict]:
    diagnostics = bc.fit_diagnostics(f"recovery:{model_name}", idata, az)
    rows = []
    for parameter, truth in truths.items():
        posterior = idata.posterior[parameter].values.ravel()
        low, high = np.quantile(posterior, [0.025, 0.975])
        rows.append({
            "recovery_version": RECOVERY_VERSION,
            "model": model_name, "parameter": parameter, "truth": truth,
            "median": float(np.median(posterior)), "hdi_low": float(low),
            "hdi_high": float(high), "covered": bool(low <= truth <= high),
            "diagnostics_ok": diagnostics["diagnostics_ok"],
            "max_rhat": diagnostics["max_rhat"],
            "min_ess_bulk": diagnostics["min_ess_bulk"],
            "min_ess_tail": diagnostics["min_ess_tail"],
            "divergences": diagnostics["divergences"],
        })
    return rows

def _sample_recovery(model, label: str):
    return sample(model, f"recovery:{label}", draws=RECOVERY_DRAWS,
                  tune=RECOVERY_TUNE, chains=CHAINS)

def simulate_A(df: pd.DataFrame, seed: int = SEED) -> tuple[pd.DataFrame, dict]:
    rng = np.random.default_rng(seed)
    out = df[df["bg_id"] < 3].copy().reset_index(drop=True)
    config_codes, config_levels = _codes(out["model"])
    harmonic_codes, harmonic_levels = _codes(out["f_lock"].round(3).astype(str))
    background_codes, background_levels = _codes(
        out["generator"] + "#" + out["bg_id"].astype(str))
    beta, delta_overlap, delta_patch = -0.40, 0.25, -0.15
    overlap = _overlap_scaled(out, config_levels)
    log_patch = _logP_centred(out, config_levels)
    mu = (beta + delta_overlap * overlap[config_codes] + delta_patch * log_patch[config_codes]
          + rng.normal(0, .10, len(config_levels))[config_codes]
          + rng.normal(0, .12, len(harmonic_levels))[harmonic_codes]
          + rng.normal(0, .08, len(background_levels))[background_codes])
    out["d"] = mu + .25 * stats.t.rvs(NU, size=len(out),
                                       random_state=rng.integers(1 << 31))
    return out, {"beta_bar": beta, "delta_O": delta_overlap, "delta_P": delta_patch}

def simulate_B(df: pd.DataFrame, seed: int = SEED + 1) -> tuple[pd.DataFrame, dict]:
    rng = np.random.default_rng(seed)
    out = df.copy().reset_index(drop=True)
    stage_codes, stages = _codes(out["stage"])
    geometry_codes, geometries = _codes(out["model"])
    theta = 0.30
    alpha = np.log(max(float(out["L_bits"].mean()), 1.0))
    log_mu = (alpha + theta * out["is_locked"].to_numpy(float)
              + rng.normal(0, .20, len(stages))[stage_codes]
              + rng.normal(0, .12, len(geometries))[geometry_codes])
    shape = 20.0
    mean = np.exp(log_mu)
    out["L_bits"] = rng.gamma(shape, mean / shape)
    return out, {"theta_lock": theta}

def simulate_C(df: pd.DataFrame, seed: int = SEED + 2) -> tuple[pd.DataFrame, dict]:
    rng = np.random.default_rng(seed)
    out = df[df["bg_id"] < 3].copy().reset_index(drop=True)
    bins = np.floor(np.mod(out["phase"], 2 * np.pi) /
                    (2 * np.pi / N_PHASE_BINS)).astype(int)
    sigma_phase = 0.08
    offsets = np.linspace(-1, 1, N_PHASE_BINS)
    offsets = sigma_phase * (offsets - offsets.mean()) / offsets.std()
    out["y_deficit"] = 0.35 + offsets[bins] + .18 * stats.t.rvs(
        NU, size=len(out), random_state=rng.integers(1 << 31))
    return out, {"sigma_phase": sigma_phase}

def simulate_D1(df: pd.DataFrame, seed: int = SEED + 3) -> tuple[pd.DataFrame, dict]:
    rng = np.random.default_rng(seed)
    generated_mode = next(mode for mode in CFG.generators if mode in set(df["mode"]))
    out = df[(df["mode"] == generated_mode) & (df["rep"] == 0)].copy().reset_index(drop=True)
    geometry_codes, geometries = _codes(out["model"])
    on_stride = np.zeros(len(out))
    on_patch = np.zeros(len(out))
    for code, _ in enumerate(geometries):
        selected = geometry_codes == code
        P = int(out.loc[selected, "P"].iloc[0])
        S = int(out.loc[selected, "S"].iloc[0])
        frequencies = out.loc[selected, "f"].to_numpy(float)
        on_stride[selected] = pl.comb_distance(frequencies, pl.FS / S) <= GRID_TOL_HZ
        on_patch[selected] = pl.comb_distance(frequencies, pl.FS / P) <= GRID_TOL_HZ
    theta_stride, theta_patch = -0.65, -0.45
    log_z = (rng.normal(0, .10, len(geometries))[geometry_codes]
             + theta_stride * on_stride + theta_patch * on_patch + rng.normal(0, .30, len(out)))
    out["z_norm"] = np.exp(log_z)
    return out, {"theta_S": theta_stride, "theta_P": theta_patch}

def simulate_D2(branch: str, seed: int) -> tuple[pd.DataFrame, dict]:
    rng = np.random.default_rng(seed)
    kappa = 0.95
    rows = []
    for P, S in FULL_MODELS:
        predicted = pl.FS / (S if branch == "stride" else P)
        for mode in CFG.generators:
            for replicate in range(3):
                measured = rng.normal(kappa * predicted,
                                      np.sqrt(0.5 ** 2 + CFG.collapse_step ** 2))
                rows.append({
                    "model": pl.model_tag(P, S), "P": P, "S": S, "mode": mode,
                    "rep": replicate, "branch": branch, "predicted_spacing": predicted,
                    "f1": measured, "delta_hat": measured, "n_sites": 3,
                })
    name = "kappa_S" if branch == "stride" else "kappa_P"
    return pd.DataFrame(rows), {name: kappa}

if have(RECOVERY_FILE):
    recovery_tbl = load_df(RECOVERY_FILE)
else:
    recovery_rows = []
    sim_A, truth_A = simulate_A(live)
    recovery_rows += _recovery_rows(
        _sample_recovery(model_A_contrast(sim_A), "A"), "A", truth_A)
    sim_B, truth_B = simulate_B(mdl_cells)
    recovery_rows += _recovery_rows(
        _sample_recovery(model_B_codelength(sim_B, adjusted=True), "B"), "B", truth_B)
    sim_C, truth_C = simulate_C(live)
    recovery_rows += _recovery_rows(
        _sample_recovery(model_C_phase(sim_C, with_phase=True), "C"), "C", truth_C)
    sim_D1, truth_D1 = simulate_D1(collapse)
    recovery_rows += _recovery_rows(
        _sample_recovery(model_D1_sites(sim_D1, "both"), "D1"), "D1", truth_D1)
    for branch, model_name, seed in (
        ("stride", "D2-stride", SEED + 4), ("patch", "D2-patch", SEED + 5)
    ):
        sim_D2, truth_D2 = simulate_D2(branch, seed)
        recovery_rows += _recovery_rows(
            _sample_recovery(model_D2_movement(sim_D2, branch), model_name),
            model_name, truth_D2)
    recovery_tbl = save_df(pd.DataFrame(recovery_rows), RECOVERY_FILE)

required_recovery_models = {"A", "B", "C", "D1", "D2-stride", "D2-patch"}
if set(recovery_tbl.get("recovery_version", [])) != {RECOVERY_VERSION}:
    raise ValueError("stale/incompatible parameter-recovery checkpoint")
RECOVERY_OK = (bc.recovery_gate(recovery_tbl, required_recovery_models)
               and recovery_tbl["diagnostics_ok"].all())
display(recovery_tbl.round(4))
print("SYNTHETIC METHOD VALIDATION ONLY - NON-REPORTABLE")
if not RECOVERY_OK:
    raise RuntimeError("parameter recovery or its convergence gate failed; empirical fitting is blocked")
print("parameter recovery gate: PASS for A, B, C, D1 and both D2 branches")

---
# Part 4, Posterior inference

Priors are fixed, the data are in, the likelihood is validated. This part fits the five models and
reports, for each, what the deliverable's *Inference* paragraph asks for: the effect on its natural
scale with a 95% credible interval, the posterior probability of the preregistered threshold, and,
where two formulations compete, a leave-one-out comparison rather than a null-hypothesis test.

Every fit is checkpointed individually, so a disconnect costs at most one model.

In [ ]:
banner("PART 4, POSTERIOR INFERENCE")

def report(idata, var: str, transform=None, label: str = "") -> dict:
    """Posterior median + 95% credible interval for one parameter, on its natural scale."""
    x = idata.posterior[var].values.ravel()
    if transform is not None:
        x = transform(x)
    lo, hi = np.quantile(x, [0.025, 0.975])
    print(f"  {label or var:<24s} median {np.median(x):+.4f}   95% CrI [{lo:+.4f}, {hi:+.4f}]")
    return dict(parameter=label or var, median=float(np.median(x)),
                lo=float(lo), hi=float(hi), sd=float(x.std()))

def prob(idata, var: str, condition) -> float:
    return float(np.mean(condition(idata.posterior[var].values.ravel())))

## 4.1, Model A: H1 at the behavioural level

**Estimand.** $\bar\beta$, the population log-ratio of forecast amplitude recovery at a phase-lock
against its matched controls. $e^{\bar\beta} < 1$ means the lock recovers *less*, which is what H1
predicts.

**Reported.** $e^{\bar\beta}$ with a 95% credible interval; $\Pr(\bar\beta<\log 0.8\mid D)$, the
probability of at least 20% attenuation; and $\Pr(|\bar\beta|<\log 1.1\mid D)$, the probability the
effect is practically nil. The per-configuration $\beta_c$ are shown as a forest plot, because H1
does not have to hold uniformly, and the overlap slope $\delta_O$ says whether it weakens as the
patches overlap more.

In [ ]:
idata_A = cached_fit("04_A.nc", lambda: sample(model_A_contrast(live), "A"))

print("\nModel A - H1 behavioural and M1")
rowsA = [report(idata_A, "beta_bar", label="beta_bar (log ratio)"),
         report(idata_A, "beta_bar", np.exp, label="recovery ratio exp(beta_bar)"),
         report(idata_A, "delta_O", label="delta_O (overlap mitigation)"),
         report(idata_A, "delta_P", label="delta_P (log patch-size slope)")]
pA_att = prob(idata_A, "beta_bar", lambda x: x < ATTENUATION_20)
pA_rope = prob(idata_A, "beta_bar", lambda x: np.abs(x) < ROPE_LOG)
pA_neg = prob(idata_A, "beta_bar", lambda x: x < 0)
pA_mit = prob(idata_A, "delta_O", lambda x: x > 0)
print(f"P(beta_bar < log 0.8 | D) = {pA_att:.3f}")
print(f"P(|beta_bar| < log 1.1 | D) = {pA_rope:.3f}")
print(f"P(beta_bar < 0 | D) = {pA_neg:.3f}")
print(f"P(delta_O > 0 | D) = {pA_mit:.3f}  (positive is mitigation on d)")

fitsM1 = {"overlap + patch size": idata_A}
for level, label in (("overlap", "overlap only"), ("patch", "patch size only"),
                     ("none", "neither")):
    fitsM1[label] = cached_fit(
        f"04_A_{level}.nc",
        lambda level=level, label=label: sample(
            model_A_contrast(live, config_level=level), f"A:{label}"),
    )

cmpM1_all, looQ_M1_all, looM1_all = bc.loo_compare(fitsM1, az)
cmpM1, looQ_M1, looM1 = bc.loo_compare({
    "overlap + patch size": fitsM1["overlap + patch size"],
    "patch size only": fitsM1["patch size only"],
}, az)
M1_LOO_WIN = bc.required_loo_win(looM1, "overlap + patch size")
display(cmpM1_all)
display(looQ_M1_all)

axes = az.plot_forest(idata_A, var_names=["beta"], combined=True, hdi_prob=0.95,
                      figsize=(8, 3.4))
axes[0].axvline(0, color="k", lw=.9)
axes[0].axvline(ATTENUATION_20, color="crimson", ls="--", lw=1)
axes[0].set_title("Model A: configuration effects")
plt.tight_layout()
plt.savefig(FIG_DIR / "P4_A_forest.png", dpi=140, bbox_inches="tight")
plt.show()

## 4.2, Model B: H1 at the representational level

**Estimand.** $\theta_{lock}$, the log-ratio by which the frequency-local prequential codelength
expands at a locked frequency. $e^{\theta_{lock}} > 1$ means the representation needs *more* bits
to separate a locked frequency from its neighbours, information loss in the sense of Voita &
Titov, independent of whether the forecast happens to recover the tone.

This is the second half of the deliverable's original two-part framework, and it can disagree with
Model A. Model A asks whether the forecast rebuilds the tone; Model B asks whether the frequency is
still *readable* inside the network. A lock that is rebuilt but no longer distinguishable from its
neighbours would show up here and nowhere else.

The unadjusted form (Eq. (10) taken literally) is fitted alongside and compared by LOO, so the
blocking decision of Ãƒâ€šSection 3.2 is reported rather than assumed.

In [ ]:
idata_B = cached_fit("04_B.nc", lambda: sample(model_B_codelength(mdl_cells, adjusted=True), "B"))
idata_B0 = cached_fit("04_B_unadjusted.nc",
                      lambda: sample(model_B_codelength(mdl_cells, adjusted=False), "B0"))

print("\nModel B - H1 representational")
rowsB = [report(idata_B, "theta_lock", label="theta_lock (log ratio)"),
         report(idata_B, "theta_lock", np.exp, label="codelength ratio"),
         report(idata_B, "k", label="Gamma shape k")]
pB_exp = prob(idata_B, "theta_lock", lambda x: x > 0)
pB_20 = prob(idata_B, "theta_lock", lambda x: x > np.log(1.2))
pB_rope = prob(idata_B, "theta_lock", lambda x: np.abs(x) < ROPE_LOG)
print(f"P(theta_lock > log 1.2 | D) = {pB_20:.3f}")
print(f"P(|theta_lock| < log 1.1 | D) = {pB_rope:.3f}")

cmpB, looQ_B, looB = bc.loo_compare({
    "stage+geometry adjusted": idata_B, "literal unadjusted": idata_B0
}, az)
display(cmpB)
display(looQ_B)

## 4.3, Model C: H2, is the deficit set by the geometry or by the phase?

**Estimand.** $\sigma_\phi$, the spread of the per-phase offsets. The phase circle is cut into
eight equal slices, each slice gets its own offset, and $\sigma_\phi$ says how far apart those
offsets are, that is, how much the lock deficit moves as the signal slides through its cycle.

H2 says the deficit is set by the stride/patch alignment and not by where the signal happens to
start, so it predicts $\sigma_\phi \approx 0$. Two readings are reported, answering different
questions:

* $\Pr(\sigma_\phi<\log 1.1\mid D)$, is the phase dependence *practically* negligible? (a
  magnitude question)
* LOO against the model with no phase term at all, does allowing phase to matter *predict better*?
  (an evidential question)

They can disagree, and the disagreement is informative: a small but consistently detected phase
dependence would show a low equivalence probability together with a LOO preference for the phase
model.

In [ ]:
idata_C = cached_fit("04_C.nc", lambda: sample(model_C_phase(live, with_phase=True), "C"))
idata_C0 = cached_fit("04_C_nophase.nc",
                      lambda: sample(model_C_phase(live, with_phase=False), "C0"))

print("\nModel C - H2 phase invariance")
rowsC = [report(idata_C, "sigma_phase", label="sigma_phase"),
         report(idata_C, "beta_bar", label="beta_bar (deficit scale)")]
pC_rope = prob(idata_C, "sigma_phase", lambda x: x < ROPE_LOG)
print(f"P(sigma_phase < log 1.1 | D) = {pC_rope:.3f}")

cmpC, looQ_C, looC = bc.loo_compare({
    "phase-dependent": idata_C, "phase-free": idata_C0
}, az)
display(cmpC)
display(looQ_C)

u_phase = idata_C.posterior["u_phase"].values.reshape(-1, N_PHASE_BINS)
low, median, high = np.quantile(u_phase, [0.025, 0.5, 0.975], axis=0)
centres = (np.arange(N_PHASE_BINS) + 0.5) * (2 * np.pi / N_PHASE_BINS)
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
ax[0].errorbar(centres, median, yerr=[median - low, high - median], fmt="o",
               color="crimson", capsize=3)
ax[0].axhline(0, color="k", lw=.8)
ax[0].set_xlabel(r"phase $\phi$ [rad]")
ax[0].set_title("per-phase offsets, 95% CrI")
sigma_post = idata_C.posterior["sigma_phase"].values.ravel()
ax[1].hist(sigma_post, bins=70, density=True, color="crimson", alpha=.8, label="posterior")
ax[1].hist(sigma_phase_prior, bins=70, density=True, histtype="step", color="0.4",
           label="prior")
ax[1].axvline(ROPE_LOG, color="k", ls="--")
ax[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "P4_C_phase.png", dpi=140, bbox_inches="tight")
plt.show()

## 4.4, Model D1: H3, which parameter generates the degradation sites?

**Estimand.** Not an effect size but a *location law*: is the token dispersion systematically lower
on one of the two predicted grids? Each frequency is labelled by the grid it falls on (within 1 Hz)
and three hypotheses are fitted to the same profiles, then compared by LOO:

* $M_S$, only the stride indicator $\mathbb{1}[f \in \{c f_s/S\}]$;
* $M_P$, only the patch indicator $\mathbb{1}[f \in \{k f_s/P\}]$;
* $M_0$, neither.

H3 predicts $\theta_S < 0$ under $M_S$. The comparison is only meaningful because the sweep grid is
the union of *every* geometry's predicted sites: each model is scored where its rival predicts a dip
as well as where it does. And it is only identified because the design contains configurations with
$S<P$, where the two grids genuinely differ, on the $P=S$ diagonal they coincide by construction
and no amount of data can separate them.

Fitted per signal mode. On a pure sinusoid the dispersion is exactly zero at a lock, so `z_norm` is
floored at `COMB_FLOOR` before the log; the floor bounds how large $|\theta_S|$ can get but affects
$M_S$, $M_P$ and $M_0$ identically, leaving the *comparison*, which is the inference, untouched.

In [ ]:
D1 = {}
cmpD1_rows = []
cmpD1_decisions = {}
looQ_D1 = {}
D1_BOTH_LABEL = "M_SP: both grids (H3)"

for mode in sorted(collapse["mode"].unique()):
    subset = collapse[collapse["mode"] == mode].reset_index(drop=True)
    fits = {}
    for grid, label in (("both", D1_BOTH_LABEL), ("stride", "M_S: stride grid"),
                        ("patch", "M_P: patch grid"), ("none", "M_0: no grid")):
        checkpoint = f"04_D1_{mode}_{grid}.nc"
        fits[label] = cached_fit(
            checkpoint,
            lambda grid=grid, subset=subset, mode=mode: sample(
                model_D1_sites(subset, grid), f"D1:{mode}:{grid}"),
        )
    D1[mode] = fits
    comparison, quality, decision = bc.loo_compare(fits, az)
    cmpD1_decisions[mode] = decision
    looQ_D1[mode] = quality
    comparison.insert(0, "mode", mode)
    cmpD1_rows.append(comparison.reset_index().rename(columns={"index": "model"}))
    print(f"\n--- collapse mode: {mode} ---")
    display(comparison)
    display(quality)

cmpD1 = pd.concat(cmpD1_rows, ignore_index=True)
display(cmpD1.sort_values(["mode", "rank"]))
print("Pure is an exact-degeneracy reference. Only TSMixup and KernelSynth enter the empirical H3 verdict.")

In [ ]:
# the observed profiles with both candidate grids drawn on: this is where H3 is visible or not
modes = sorted(collapse["mode"].unique())
geoms = [pl.model_tag(P, S) for P, S in pl.MODELS]
fig, axes = plt.subplots(len(geoms), len(modes), figsize=(5.0 * len(modes), 2.0 * len(geoms)),
                         sharex=True, squeeze=False)
for r, gname in enumerate(geoms):
    for c, mode in enumerate(modes):
        ax = axes[r][c]
        sub = collapse[(collapse["model"] == gname) & (collapse["mode"] == mode)]
        if len(sub) == 0:
            ax.axis("off"); continue
        prof = sub.groupby("f", as_index=False)["z_norm"].mean().sort_values("f")
        P0, S0 = int(sub["P"].iloc[0]), int(sub["S"].iloc[0])
        ax.plot(prof["f"], prof["z_norm"], color="#c62828", lw=1.0)
        for f in pl.stride_locks(S0):
            ax.axvline(f, color="#6a1b9a", ls="--", lw=.9, alpha=.65)
        for f in pl.patch_nulls(P0):
            ax.axvline(f, color="#1565c0", ls=":", lw=.9, alpha=.5)
        ax.set_yticks([]); ax.margins(x=.01)
        if c == 0:
            ax.set_ylabel(gname, rotation=0, ha="right", va="center", fontsize=9)
        if r == 0:
            ax.set_title(mode, fontsize=10)
        if r == len(geoms) - 1:
            ax.set_xlabel("frequency [Hz]")
fig.suptitle("H3: collapse profiles vs the two candidate grids "
             "(purple dashed = $c f_s/S$, blue dotted = $k f_s/P$)", y=1.005, fontsize=11)
fig.tight_layout(); fig.savefig(FIG_DIR / "P4_D1_combs.png", dpi=140, bbox_inches="tight"); plt.show()

## 4.5, Models D2: H3a and H3b, do the sites *move* with the geometry?

$\S 4.4$ says where the dips sit. It does not yet say they *move*: each geometry's grid is fixed,
so a model fitting that grid well is still consistent with dips that merely happen to coincide with
it. H3's actual claim is a law across geometries, and it is **two** laws, one per branch.

$\mathcal F_{lock}$ is a union, so the detected set is the union of two combs and has no single
spacing. Every detected dip is therefore first assigned to the branch that predicts it, sites
belonging to both are set aside, and the fundamental of each branch is compared with the spacing
that branch predicts:

$$\hat f^{\,F}_{1,g} \;\sim\; \mathcal N\!\left(\kappa_F\,\Delta^F_g,\;
\sqrt{\sigma_F^2+\Delta f^2}\right), \qquad
\Delta^S_g = f_s/S_g, \quad \Delta^P_g = f_s/P_g .$$

**H3a predicts $\kappa_S = 1$ and H3b predicts $\kappa_P = 1$**: each branch tracks its own
parameter one for one, so a site that stays put while that parameter changes refutes its branch.

$\kappa_S$ is identified by the series at fixed $P$, which the design supplies at $P=16$, $P=24$ and $P=32$; $\kappa_P$ by the pairs that share a stride and differ in patch size, at $S=8$, $12$ and
$16$.

The residual carries a floor $\Delta f$, the sweep step: a spacing read off a discrete grid is no
more precise than one step. Without it, sites landing exactly on the prediction drive $\sigma_F$ to
zero and the posterior develops a funnel NUTS cannot traverse. With it, a near-zero $\sigma_F$ is a
result rather than a sampler failure.

Both the fundamental and the median gap are fitted. With three or four sites in band a single
spurious detection inserts a short interval and flips the median, to which the fundamental is
immune, so a disagreement between the two is itself informative about detection stability.

In [ ]:
# D2 primary response: the branch fundamental needs one detected site. The median-gap robustness
# response needs at least two. Pure-tone rows remain a reference and do not enter primary evidence.
D2_IDENTIFICATION = bc.identified_branches(
    sites, minimum_sites=CFG.min_d2_sites, modes=tuple(CFG.generators))
sites_f1 = sites[(sites["rep"] >= 0) & (sites["n_sites"] >= 1)
                 & sites["mode"].isin(CFG.generators)].copy()
sites_gap = sites[(sites["rep"] >= 0) & (sites["n_sites"] >= 2)
                  & sites["delta_hat"].notna() & sites["mode"].isin(CFG.generators)].copy()
print("D2 identification (minimum 10 unambiguous sites across generated-background design):",
      D2_IDENTIFICATION)

idata_D2, idata_D2d = {}, {}
for branch in ("stride", "patch"):
    if not D2_IDENTIFICATION[branch]:
        print(f"{branch}: NOT IDENTIFIED; no D2 posterior will be fitted")
        continue
    branch_f1 = sites_f1[sites_f1["branch"] == branch]
    if branch_f1.empty:
        raise ValueError(f"{branch} passed the site-count bar but has no replicate-level f1 rows")
    idata_D2[branch] = cached_fit(
        f"04_D2_{branch}.nc",
        lambda branch=branch: sample(model_D2_movement(sites_f1, branch, response="f1"),
                                     f"D2:{branch}:f1"),
    )
    branch_gap = sites_gap[sites_gap["branch"] == branch]
    if not branch_gap.empty:
        idata_D2d[branch] = cached_fit(
            f"04_D2_{branch}_deltahat.nc",
            lambda branch=branch: sample(
                model_D2_movement(sites_gap, branch, response="delta_hat"),
                f"D2:{branch}:delta_hat"),
        )

names = {"stride": "kappa_S", "patch": "kappa_P"}
pD2, pD2_gap = {}, {}
for branch, idata in idata_D2.items():
    parameter = names[branch]
    report(idata, parameter, label=f"{parameter} primary f1")
    pD2[branch] = prob(idata, parameter, lambda x: np.abs(x - 1.0) < ROPE_SLOPE)
    print(f"P(|{parameter}-1| < {ROPE_SLOPE} | D) = {pD2[branch]:.3f}")
    if branch in idata_D2d:
        pD2_gap[branch] = prob(idata_D2d[branch], parameter,
                               lambda x: np.abs(x - 1.0) < ROPE_SLOPE)
        print(f"median-gap robustness probability = {pD2_gap[branch]:.3f}; "
              f"absolute difference={abs(pD2[branch] - pD2_gap[branch]):.3f}")
    else:
        print("median-gap robustness: unavailable (fewer than two sites per contributing row)")

---
# Part 5, Checks

A posterior is a conditional statement: *given* that the sampler converged, that the model can
reproduce the data, and that the answer does not hinge on an arbitrary prior scale. This part tests
all three, and only then states the verdicts.

* **5.1 Convergence**, $\hat R$, bulk and tail ESS, divergences, against the thresholds the
  deliverable preregisters ($\hat R<1.01$, ESS $>1000$, zero divergences).
* **5.2 Posterior predictive**, can the fitted model generate data that look like the observations,
  stratified by frequency, phase, $P$, $S$ and generator? A model that fits the pooled histogram but
  fails a stratum is fitting the average and missing the structure.
* **5.3 Prior sensitivity**, the whole ladder $\{0.25, 0.5, 1.0\}$ plus a Gaussian likelihood. If
  the conclusion moves with the prior scale, it was the prior's conclusion.
* **5.4 Model comparison**, the LOO tables gathered in one place.
* **5.5 Verdicts**, one row per hypothesis, next to the frequentist PASS/FAIL that
  `chronos/testing/hypotheses.py` produces on the same phenomenon.

In [ ]:
banner("PART 5 - SCIENTIFIC VALIDITY CHECKS")

FITS = {
    "A": idata_A,
    "A:overlap-only": fitsM1["overlap only"],
    "A:patch-only": fitsM1["patch size only"],
    "A:neither": fitsM1["neither"],
    "B": idata_B, "B:unadjusted": idata_B0,
    "C": idata_C, "C:phase-free": idata_C0,
}
for mode, fits in D1.items():
    for label, idata in fits.items():
        FITS[f"D1:{mode}:{label}"] = idata
for branch, idata in idata_D2.items():
    FITS[f"D2:{branch}:f1"] = idata
for branch, idata in idata_D2d.items():
    FITS[f"D2:{branch}:delta_hat"] = idata

diagnostics = save_df(bc.diagnostics_table(FITS, az), "05_diagnostics.parquet")
display(diagnostics.round(3))
failed = diagnostics.loc[~diagnostics["diagnostics_ok"], "fit"].tolist()
print("diagnostic thresholds: R-hat < 1.01, bulk/tail ESS > 1000, zero divergences")
print("diagnostics gate:", "PASS" if not failed else f"FAIL for {failed}")

def diagnostics_ok(*names: str) -> bool:
    subset = diagnostics[diagnostics["fit"].isin(names)]
    return bool(len(subset) == len(set(names)) and subset["diagnostics_ok"].all())

## 5.2 - Posterior predictive checks, pooled and stratified

The primary A, B, C, D1 and D2 fits are checked at the level at which each model can fail. At least
90% of preregistered strata must place the observed mean inside the 95% replicated-mean interval.
This check is a model adequacy gate, not a hypothesis result.

In [ ]:
PPC_VERSION = "d3-all-primary-v1"
PPC_FILE = "05_ppc_d3_v1.parquet"

def _ppc_rows(analysis: str, model, idata, observed_name: str, observed,
              frame: pd.DataFrame, columns: tuple[str, ...]) -> tuple[list[dict], set[str]]:
    frame = frame.reset_index(drop=True)
    observed = np.asarray(observed, float)
    if len(frame) != len(observed):
        raise ValueError(f"{analysis}: PPC frame/observation length mismatch")
    with model:
        predictive = pm.sample_posterior_predictive(idata, random_seed=SEED, progressbar=False)
    replicated = np.asarray(predictive.posterior_predictive[observed_name]).reshape(-1, len(frame))
    rows, required = [], set()
    for column in columns:
        for level, positions in frame.groupby(column, observed=True).indices.items():
            positions = np.asarray(positions, int)
            observed_mean = float(observed[positions].mean())
            replicated_means = replicated[:, positions].mean(axis=1)
            low, high = np.quantile(replicated_means, [0.025, 0.975])
            stratum = f"{analysis}:{column}={level}"
            required.add(stratum)
            rows.append({
                "ppc_version": PPC_VERSION, "analysis": analysis, "stratum": stratum,
                "n": len(positions), "observed": observed_mean,
                "rep_low": float(low), "rep_high": float(high),
                "ppc_ok": bool(low <= observed_mean <= high),
            })
    return rows, required

if have(PPC_FILE):
    ppc_table = load_df(PPC_FILE)
    required_ppc = set(ppc_table["stratum"])
else:
    ppc_rows, required_ppc = [], set()
    rows, required = _ppc_rows(
        "A", model_A_contrast(live), idata_A, "d", live["d"].to_numpy(),
        live, ("model", "generator"))
    ppc_rows += rows; required_ppc |= required

    rows, required = _ppc_rows(
        "B", model_B_codelength(mdl_cells, adjusted=True), idata_B, "y",
        np.clip(mdl_cells["L_bits"].to_numpy(float), 1e-3, None),
        mdl_cells, ("model", "stage"))
    ppc_rows += rows; required_ppc |= required

    rows, required = _ppc_rows(
        "C", model_C_phase(live, with_phase=True), idata_C, "y",
        live["y_deficit"].to_numpy(float), live, ("model", "generator"))
    ppc_rows += rows; required_ppc |= required

    for mode in CFG.generators:
        subset = collapse[collapse["mode"] == mode].reset_index(drop=True)
        rows, required = _ppc_rows(
            "D1", model_D1_sites(subset, "both"), D1[mode][D1_BOTH_LABEL], "logz",
            np.log(np.clip(subset["z_norm"].to_numpy(float), COMB_FLOOR, None)),
            subset, ("model",))
        ppc_rows += rows; required_ppc |= required

    for branch, idata in idata_D2.items():
        subset = sites_f1[sites_f1["branch"] == branch].reset_index(drop=True)
        rows, required = _ppc_rows(
            f"D2-{branch}", model_D2_movement(sites_f1, branch, response="f1"),
            idata, "f1_hat", subset["f1"].to_numpy(float), subset, ("model", "mode"))
        ppc_rows += rows; required_ppc |= required

    ppc_table = save_df(pd.DataFrame(ppc_rows), PPC_FILE)

if set(ppc_table.get("ppc_version", [])) != {PPC_VERSION}:
    raise ValueError("stale/incompatible PPC checkpoint")
PPC_OK = bc.ppc_gate(ppc_table, required_ppc, minimum_coverage=0.90)
display(ppc_table)
display(ppc_table.groupby("analysis")["ppc_ok"].agg(["sum", "count", "mean"]))
print("posterior-predictive gate:", "PASS" if PPC_OK else "FAIL - claims become NOT REPORTABLE")

plot_table = ppc_table.sort_values(["analysis", "stratum"]).reset_index(drop=True)
figure, axis = plt.subplots(figsize=(10, max(4, .18 * len(plot_table))))
y = np.arange(len(plot_table))
axis.hlines(y, plot_table["rep_low"], plot_table["rep_high"], color="steelblue", lw=3, alpha=.6)
axis.scatter(plot_table["observed"], y, c=np.where(plot_table["ppc_ok"], "black", "crimson"), s=12)
axis.set_yticks(y)
axis.set_yticklabels(plot_table["stratum"], fontsize=6)
axis.set_title("Posterior predictive stratum means (red = outside 95% replicated interval)")
figure.tight_layout()
figure.savefig(FIG_DIR / "P5_ppc.png", dpi=140, bbox_inches="tight")
plt.show()

## 5.3 - Prior-scale and likelihood sensitivity

Model A is refitted at all three preregistered prior scales and under the Gaussian likelihood.
Both the H1 attenuation probability and M1 positive-overlap probability must vary by at most 0.10,
and every sensitivity fit must pass the full convergence gate.

In [ ]:
SENSITIVITY_VERSION = "d3-a-full-v1"
SENSITIVITY_FILE = "05_sensitivity_d3_v1.parquet"

if have(SENSITIVITY_FILE):
    sens = load_df(SENSITIVITY_FILE)
else:
    rows = []
    variants = [(f"StudentT scale {scale}", scale, "student") for scale in PRIOR_LADDER]
    variants.append(("Normal likelihood scale 0.5", 0.5, "normal"))
    for index, (label, scale, likelihood) in enumerate(variants):
        fit = cached_fit(
            f"05_sensitivity_{index}.nc",
            lambda label=label, scale=scale, likelihood=likelihood: sample(
                model_A_contrast(live, scale=scale, likelihood=likelihood),
                f"sensitivity:{label}"),
        )
        beta = fit.posterior["beta_bar"].values.ravel()
        delta = fit.posterior["delta_O"].values.ravel()
        diagnostic = bc.fit_diagnostics(f"sensitivity:{label}", fit, az)
        rows.append({
            "sensitivity_version": SENSITIVITY_VERSION, "variant": label,
            "median_beta": float(np.median(beta)),
            "beta_low": float(np.quantile(beta, .025)), "beta_high": float(np.quantile(beta, .975)),
            "P_attenuation_20": float(np.mean(beta < ATTENUATION_20)),
            "P_beta_in_rope": float(np.mean(np.abs(beta) < ROPE_LOG)),
            "P_delta_O_positive": float(np.mean(delta > 0)),
            "diagnostics_ok": diagnostic["diagnostics_ok"],
            "max_rhat": diagnostic["max_rhat"],
            "min_ess_bulk": diagnostic["min_ess_bulk"],
            "min_ess_tail": diagnostic["min_ess_tail"],
            "divergences": diagnostic["divergences"],
        })
    sens = save_df(pd.DataFrame(rows), SENSITIVITY_FILE)

if set(sens.get("sensitivity_version", [])) != {SENSITIVITY_VERSION}:
    raise ValueError("stale/incompatible sensitivity checkpoint")
SENSITIVITY_OK = (bc.sensitivity_gate(
    sens, ("P_attenuation_20", "P_delta_O_positive"), max_spread=0.10)
    and sens["diagnostics_ok"].all())
display(sens.round(4))
print("sensitivity gate:", "PASS" if SENSITIVITY_OK else "FAIL - A/M1 become NOT REPORTABLE")

figure, axis = plt.subplots(figsize=(8, 3.2))
y = np.arange(len(sens))
axis.hlines(y, sens["beta_low"], sens["beta_high"], color="steelblue", lw=4, alpha=.65)
axis.scatter(sens["median_beta"], y, color="crimson")
axis.axvline(ATTENUATION_20, color="crimson", ls="--")
axis.axvline(0, color="k", lw=.8)
axis.set_yticks(y)
axis.set_yticklabels(sens["variant"])
axis.set_title("Model A prior/likelihood sensitivity")
figure.tight_layout()
figure.savefig(FIG_DIR / "P5_sensitivity.png", dpi=140, bbox_inches="tight")
plt.show()

## 5.4 - LOO comparisons with Pareto-k reliability

A comparison is decisive only when every PSIS-LOO object is reliable and the best model leads the
runner-up by at least twice the reported standard error. H2 reports its magnitude and LOO reading
separately because they answer different questions and may legitimately disagree.

In [ ]:
def loo_reading(decision: dict) -> str:
    if not decision["loo_reliable"]:
        return "INCONCLUSIVE: unreliable PSIS-LOO (inspect Pareto k)"
    if not decision["separated_2dse"]:
        return (f"INCONCLUSIVE: margin {decision['elpd_margin']:.2f} < "
                f"2*dse {2 * decision['dse']:.2f}")
    return f"{decision['winner']} preferred"

print("M1 exact comparison - full overlap+patch vs patch-only")
display(cmpM1); display(looQ_M1); print(loo_reading(looM1))
print("\nM1 four-way descriptive comparison")
display(cmpM1_all); display(looQ_M1_all); print(loo_reading(looM1_all))
print("\nModel B blocking comparison")
display(cmpB); display(looQ_B); print(loo_reading(looB))
print("\nH2 phase-dependent vs phase-free (separate from sigma_phase magnitude)")
display(cmpC); display(looQ_C); print(loo_reading(looC))
print("\nH3 D1 comparisons")
for mode in sorted(cmpD1_decisions):
    print(mode, "-", loo_reading(cmpD1_decisions[mode]))
    display(looQ_D1[mode])

## 5.5 - Fail-closed Deliverable 3 verdicts

Only empirical full-design fits can populate this table. Diagnostics, parameter recovery,
posterior-predictive adequacy and the applicable sensitivity/LOO checks are explicit gates. A
failed gate yields `NOT REPORTABLE`; a D2 branch below ten unambiguous sites yields
`NOT IDENTIFIED`, never a null result. No static result from another notebook or frequentist run is
copied into this table.

In [ ]:
def recovery_ok(model_name: str) -> bool:
    subset = recovery_tbl[recovery_tbl["model"] == model_name]
    return bool(not subset.empty and subset["covered"].all() and subset["diagnostics_ok"].all())

def ppc_ok(analysis: str) -> bool:
    subset = ppc_table[ppc_table["analysis"] == analysis]
    return bool(not subset.empty and subset["ppc_ok"].mean() >= 0.90)

verdict_rows = []

gate_A = diagnostics_ok("A") and recovery_ok("A") and ppc_ok("A") and SENSITIVITY_OK
verdict_rows.append({
    "hypothesis": "H1 behavioural", "model": "A",
    "decision_rule": "P(beta_bar < log 0.8)>=.95; refute by beta ROPE>=.95",
    "probability": pA_att, "refute_probability": pA_rope,
    "gate_ok": gate_A,
    "verdict": bc.three_way_verdict(pA_att, pA_rope, gate_A),
    "comparison_or_robustness": "prior/likelihood probability spread <=0.10",
})

gate_B = diagnostics_ok("B") and recovery_ok("B") and ppc_ok("B")
verdict_rows.append({
    "hypothesis": "H1 representational", "model": "B",
    "decision_rule": "P(theta_lock > log 1.2)>=.95; refute by theta ROPE>=.95",
    "probability": pB_20, "refute_probability": pB_rope,
    "gate_ok": gate_B,
    "verdict": bc.three_way_verdict(pB_20, pB_rope, gate_B),
    "comparison_or_robustness": loo_reading(looB),
})

gate_C = diagnostics_ok("C") and recovery_ok("C") and ppc_ok("C")
verdict_rows.append({
    "hypothesis": "H2 phase invariance", "model": "C",
    "decision_rule": "P(sigma_phase < log 1.1)>=.95",
    "probability": pC_rope, "refute_probability": 1 - pC_rope,
    "gate_ok": gate_C,
    "verdict": bc.three_way_verdict(pC_rope, 1 - pC_rope, gate_C),
    "comparison_or_robustness": loo_reading(looC),
})

generated_modes = tuple(CFG.generators)
d1_fit_names = [f"D1:{mode}:{label}" for mode in generated_modes for label in D1[mode]]
d1_sign_probabilities = {}
d1_mode_support = {}
for mode in generated_modes:
    primary = D1[mode][D1_BOTH_LABEL]
    p_stride_negative = prob(primary, "theta_S", lambda x: x < 0)
    p_patch_negative = prob(primary, "theta_P", lambda x: x < 0)
    d1_sign_probabilities[mode] = min(p_stride_negative, p_patch_negative)
    d1_mode_support[mode] = (
        bc.required_loo_win(cmpD1_decisions[mode], D1_BOTH_LABEL)
        and p_stride_negative >= .95 and p_patch_negative >= .95)
gate_D1 = (diagnostics_ok(*d1_fit_names) and recovery_ok("D1") and ppc_ok("D1")
           and all(cmpD1_decisions[mode]["loo_reliable"] for mode in generated_modes))
d1_probability = min(d1_sign_probabilities.values())
if not gate_D1:
    d1_verdict = "NOT REPORTABLE"
elif all(d1_mode_support.values()):
    d1_verdict = "supported"
elif any(d1_probability <= .05 or
         (cmpD1_decisions[mode]["comparison_ok"] and
          cmpD1_decisions[mode]["winner"] != D1_BOTH_LABEL) for mode in generated_modes):
    d1_verdict = "refuted"
else:
    d1_verdict = "inconclusive"
verdict_rows.append({
    "hypothesis": "H3 site location", "model": "D1",
    "decision_rule": "both-grid fit wins LOO and theta_S, theta_P < 0 on both generators",
    "probability": d1_probability, "refute_probability": np.nan,
    "gate_ok": gate_D1, "verdict": d1_verdict,
    "comparison_or_robustness": "; ".join(
        f"{mode}: {loo_reading(cmpD1_decisions[mode])}" for mode in generated_modes),
})

gate_M1 = (diagnostics_ok("A", "A:patch-only") and recovery_ok("A") and ppc_ok("A")
           and SENSITIVITY_OK and looM1["loo_reliable"])
if not gate_M1:
    m1_verdict = "NOT REPORTABLE"
elif pA_mit >= .95 and M1_LOO_WIN:
    m1_verdict = "supported"
elif pA_mit <= .05:
    m1_verdict = "refuted"
else:
    m1_verdict = "inconclusive"
verdict_rows.append({
    "hypothesis": "M1 overlap mitigation", "model": "A'",
    "decision_rule": "P(delta_O > 0)>=.95 and full model beats patch-only by >=2*dse",
    "probability": pA_mit, "refute_probability": 1 - pA_mit,
    "gate_ok": gate_M1, "verdict": m1_verdict,
    "comparison_or_robustness": loo_reading(looM1),
})

for branch, hypothesis, parameter in (
    ("stride", "H3a stride movement", "kappa_S"),
    ("patch", "H3b patch movement", "kappa_P"),
):
    identified = D2_IDENTIFICATION[branch]
    if not identified:
        row = {"probability": np.nan, "refute_probability": np.nan, "gate_ok": True,
               "verdict": "NOT IDENTIFIED", "comparison_or_robustness": "<10 sites"}
    else:
        primary_name = f"D2:{branch}:f1"
        gap_names = ([f"D2:{branch}:delta_hat"] if branch in idata_D2d else [])
        gate = (diagnostics_ok(primary_name, *gap_names)
                and recovery_ok(f"D2-{branch}") and ppc_ok(f"D2-{branch}"))
        probability = pD2[branch]
        robustness = (f"gap probability={pD2_gap[branch]:.3f}, "
                      f"difference={abs(probability - pD2_gap[branch]):.3f}"
                      if branch in pD2_gap else "gap robustness unavailable")
        row = {
            "probability": probability, "refute_probability": 1 - probability,
            "gate_ok": gate,
            "verdict": bc.three_way_verdict(probability, 1 - probability, gate),
            "comparison_or_robustness": robustness,
        }
    verdict_rows.append({
        "hypothesis": hypothesis, "model": "D2",
        "decision_rule": f"P(|{parameter}-1| < {ROPE_SLOPE})>=.95", **row,
    })

verdicts = save_df(pd.DataFrame(verdict_rows), "05_verdicts.parquet")
display(verdicts)
print("\nPure-tone/smoke/recovery output is intentionally absent from empirical verdicts.")
print("checkpoints:", CKPT_DIR)

---
## Scope of the conclusions

Every reported configuration was retrained from scratch for the same 100,000-step budget. The
remaining design limitation is the single training seed: posterior uncertainty is conditional on
one checkpoint per geometry, while the configuration hierarchy can absorb but cannot separately
identify between-seed training variation. Failed diagnostics, recovery, PPC, sensitivity or LOO
quality do not become weak evidence; they become `NOT REPORTABLE`.